# AI & Tech-Aware DCF Valuation from SEC Filings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/dcf-valuation/blob/main/DCF_Valuation_AI_Tech.ipynb)

Different businesses need different valuation models. This notebook first **classifies** the company, then picks the model:

| Company type | How it is detected | Model used |
|---|---|---|
| **AI / hyper-growth tech** | tech industry code or high R&D, plus AI-heavy 10-K or 30%+ growth | Growth-stage DCF, 10 years, 5-year high-growth stage |
| **Growth tech** | tech, 10–30% growth | Growth-stage DCF, 10 years, 3-year high-growth stage |
| **Mature tech / Traditional** | everything else | Standard 5-year DCF |

**Works for non-US companies too:** US-listed foreign companies (ASML, SAP, Novo Nordisk) are read from their 20-F (IFRS, in their own currency);
companies listed only in Europe or elsewhere (`ASML.AS`, `MC.PA`, `SIE.DE`) fall back to Yahoo Finance statements.

Both models are always run side by side, together with a **reverse DCF** (the growth rate the market is pricing in) and trading multiples, so you can see *why* they differ.

> `Runtime → Run all`. Change the inputs in step 1 and re-run.

## 0 · Setup
Run these cells once. They install two packages and create the model's code files (`dcf/…py`). You don't need to edit them.

In [ ]:
!pip install -q yfinance openpyxl
!mkdir -p dcf

In [ ]:
%%writefile dcf/__init__.py
"""SEC-powered DCF valuation: ticker in, intrinsic value per share out."""
from .financials import Financials, load_financials
from .market import MarketData, fetch_market_data
from .model import (Assumptions, DCFResult, default_assumptions, run_dcf, scenarios,
                    sensitivity, wacc_growth_grid)
from .sec import SECClient
from .classify import Classification, classify, fetch_10k_text
from .yahoo import load_company, load_financials_yahoo
from .growth import (GrowthAssumptions, growth_assumptions, growth_margin_grid, implied_growth,
                     rule_of_40, run_growth_dcf, trading_multiples, weighted_scenarios)

__version__ = "2.1.0"


def value_company(ticker: str, user_agent: str, use_ttm: bool = True,
                  market_overrides: dict | None = None, **assumption_overrides):
    """One-call pipeline: SEC filings -> market data -> assumptions -> DCF."""
    fin, _ = load_company(ticker, user_agent, include_ttm=use_ttm)
    mkt = fetch_market_data(fin.ticker, reporting_currency=fin.currency, **(market_overrides or {}))
    a = default_assumptions(fin, mkt, use_ttm=use_ttm, **assumption_overrides)
    return fin, mkt, run_dcf(fin, mkt, a, use_ttm=use_ttm)


def auto_value(ticker: str, user_agent: str, use_ttm: bool = True, mode: str = "auto",
               scan_10k: bool = True, **overrides):
    """Classify the company, then run the growth-stage DCF (AI/tech) or the standard DCF."""
    from .market import fetch_consensus_growth
    fin, client = load_company(ticker, user_agent, include_ttm=use_ttm)
    text = fetch_10k_text(client, fin) if scan_10k else None
    cls = classify(fin, text)
    mkt = fetch_market_data(fin.ticker, reporting_currency=fin.currency)
    model = cls.model if mode == "auto" else mode
    if model == "growth":
        cg, note = fetch_consensus_growth(fin.ticker)
        a = growth_assumptions(fin, mkt, cls, use_ttm, cg, note, **overrides)
        res = run_growth_dcf(fin, mkt, a, use_ttm)
    else:
        a = default_assumptions(fin, mkt, use_ttm=use_ttm, **overrides)
        res = run_dcf(fin, mkt, a, use_ttm)
    return fin, cls, mkt, res


__all__ = [
    "SECClient", "Financials", "load_financials", "MarketData", "fetch_market_data",
    "Assumptions", "DCFResult", "default_assumptions", "run_dcf", "sensitivity",
    "wacc_growth_grid", "scenarios", "value_company", "Classification", "classify",
    "fetch_10k_text", "GrowthAssumptions", "growth_assumptions", "run_growth_dcf",
    "weighted_scenarios", "growth_margin_grid", "implied_growth", "trading_multiples", "rule_of_40",
    "auto_value", "load_company", "load_financials_yahoo",
]


In [ ]:
%%writefile dcf/sec.py
"""
Thin client for the free SEC EDGAR APIs.

Endpoints used (no API key needed, but the SEC requires a descriptive
User-Agent with a contact e-mail, and max ~10 requests/second):

* https://www.sec.gov/files/company_tickers.json            ticker -> CIK
* https://data.sec.gov/submissions/CIK##########.json       filing index, SIC code
* https://data.sec.gov/api/xbrl/companyfacts/CIK##########.json
                                                             every XBRL fact ever filed
"""
from __future__ import annotations

import time
from dataclasses import dataclass
from functools import lru_cache

import requests

TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SUBMISSIONS_URL = "https://data.sec.gov/submissions/CIK{cik}.json"
COMPANYFACTS_URL = "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
ARCHIVE_URL = "https://www.sec.gov/Archives/edgar/data/{cik}/{accn}/{doc}"


class SECError(RuntimeError):
    pass


@dataclass
class Filing:
    form: str
    filing_date: str
    report_date: str
    accession: str
    url: str


class SECClient:
    """Minimal, polite EDGAR client with in-memory caching."""

    def __init__(self, user_agent: str, min_interval: float = 0.15):
        if not user_agent or "@" not in user_agent or "example.com" in user_agent:
            raise SECError(
                "The SEC requires a real User-Agent, e.g. "
                "'Jane Doe jane.doe@gmail.com'. Set SEC_USER_AGENT in the notebook."
            )
        self.session = requests.Session()
        self.session.headers.update(
            {"User-Agent": user_agent, "Accept-Encoding": "gzip, deflate"}
        )
        self._min_interval = min_interval
        self._last_call = 0.0

    # ------------------------------------------------------------------ HTTP
    def _get_json(self, url: str) -> dict:
        wait = self._min_interval - (time.time() - self._last_call)
        if wait > 0:
            time.sleep(wait)
        resp = self.session.get(url, timeout=30)
        self._last_call = time.time()
        if resp.status_code == 404:
            raise SECError(f"Not found on EDGAR: {url}")
        if resp.status_code == 403:
            raise SECError("SEC refused the request (403). Check your User-Agent string.")
        resp.raise_for_status()
        return resp.json()

    def get_text(self, url: str) -> str:
        """Raw text of a filing document (HTML), respecting the SEC rate limit."""
        wait = self._min_interval - (time.time() - self._last_call)
        if wait > 0:
            time.sleep(wait)
        resp = self.session.get(url, timeout=60)
        self._last_call = time.time()
        resp.raise_for_status()
        return resp.text

    # --------------------------------------------------------------- lookups
    @lru_cache(maxsize=1)
    def ticker_map(self) -> dict[str, dict]:
        raw = self._get_json(TICKERS_URL)
        return {
            row["ticker"].upper(): {"cik": int(row["cik_str"]), "name": row["title"]}
            for row in raw.values()
        }

    def lookup(self, ticker: str) -> dict:
        key = ticker.upper().strip().replace(".", "-")
        info = self.ticker_map().get(key)
        if info is None:
            raise SECError(
                f"Ticker '{ticker}' not found in SEC's list. Only SEC-registered "
                "companies are covered (US listings incl. ADRs that file with the SEC)."
            )
        return {"ticker": key, **info}

    @lru_cache(maxsize=32)
    def company_facts(self, cik: int) -> dict:
        return self._get_json(COMPANYFACTS_URL.format(cik=f"{cik:010d}"))

    @lru_cache(maxsize=32)
    def submissions(self, cik: int) -> dict:
        return self._get_json(SUBMISSIONS_URL.format(cik=f"{cik:010d}"))

    def latest_filing(self, cik: int, forms=("10-K", "10-K/A", "20-F", "20-F/A", "40-F")) -> Filing | None:
        """Most recent filing of the given form type, with a link to the document."""
        recent = self.submissions(cik).get("filings", {}).get("recent", {})
        for i, form in enumerate(recent.get("form", [])):
            if form in forms:
                accn = recent["accessionNumber"][i]
                return Filing(
                    form=form,
                    filing_date=recent["filingDate"][i],
                    report_date=recent["reportDate"][i],
                    accession=accn,
                    url=ARCHIVE_URL.format(
                        cik=cik, accn=accn.replace("-", ""), doc=recent["primaryDocument"][i]
                    ),
                )
        return None


In [ ]:
%%writefile dcf/financials.py
"""
Turn raw SEC XBRL "company facts" into clean, standardized financial statements.

Why this is non-trivial
-----------------------
* Every 10-K repeats prior years as comparatives, so the same fiscal year shows up
  in several filings (sometimes restated). We key every value on its *period end
  date* and keep the most recently filed number.
* Companies tag the same line item differently, and switch tags over time
  (e.g. ``SalesRevenueNet`` -> ``RevenueFromContractWithCustomerExcludingAssessedTax``).
  Each standardized line item has an ordered list of candidate US-GAAP tags,
  merged per period.
* 10-Ks also contain 3-month (Q4) facts. Flow items are restricted to ~12-month
  durations so quarters never leak into annual numbers.
* Trailing-twelve-months (TTM) = last fiscal year + current YTD - prior-year YTD,
  using the latest 10-Q, so the model can run on the freshest data available.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from .sec import Filing, SECClient

ANNUAL_FORMS = {"10-K", "10-K/A", "10-KT", "10-KT/A",      # US companies
                "20-F", "20-F/A", "40-F", "40-F/A"}       # foreign private issuers (e.g. ASML, SAP)
QUARTERLY_FORMS = {"10-Q", "10-Q/A"}

# ----------------------------------------------------------------------------
# Standardized line items -> candidate US-GAAP tags, in priority order.
# ----------------------------------------------------------------------------
FLOW_TAGS: dict[str, list[str]] = {
    # Income statement
    "revenue": [
        "Revenues",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "RevenueFromContractWithCustomerIncludingAssessedTax",
        "SalesRevenueNet",
        "SalesRevenueGoodsNet",
        "SalesRevenueServicesNet",
    ],
    "cogs": [
        "CostOfRevenue",
        "CostOfGoodsAndServicesSold",
        "CostOfGoodsSold",
        "CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization",
    ],
    "gross_profit": ["GrossProfit"],
    "rnd": ["ResearchAndDevelopmentExpense",
            "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost"],
    "sga": ["SellingGeneralAndAdministrativeExpense"],
    "operating_income": ["OperatingIncomeLoss"],
    "interest_expense": ["InterestExpense", "InterestExpenseNonoperating",
                         "InterestExpenseDebt", "InterestAndDebtExpense"],
    "pretax_income": [
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeLossFromEquityMethodInvestments",
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesDomestic",
    ],
    "income_tax": ["IncomeTaxExpenseBenefit"],
    "net_income": ["NetIncomeLoss", "ProfitLoss", "NetIncomeLossAvailableToCommonStockholdersBasic"],
    # Cash flow statement
    "da": [
        "DepreciationDepletionAndAmortization",
        "DepreciationAmortizationAndAccretionNet",
        "DepreciationAndAmortization",
        "DepreciationAmortizationAndOther",
        "Depreciation",
    ],
    "sbc": ["ShareBasedCompensation", "AllocatedShareBasedCompensationExpense"],
    "cfo": ["NetCashProvidedByUsedInOperatingActivities",
            "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"],
    "capex": ["PaymentsToAcquirePropertyPlantAndEquipment",
              "PaymentsToAcquireProductiveAssets",
              "PaymentsForCapitalImprovements"],
    "dividends": ["PaymentsOfDividends", "PaymentsOfDividendsCommonStock"],
    "buybacks": ["PaymentsForRepurchaseOfCommonStock"],
}

PER_SHARE_TAGS = {"eps_diluted": ["EarningsPerShareDiluted", "EarningsPerShareBasicAndDiluted"]}
SHARE_FLOW_TAGS = {"diluted_shares": ["WeightedAverageNumberOfDilutedSharesOutstanding",
                                      "WeightedAverageNumberOfShareOutstandingBasicAndDiluted"]}

INSTANT_TAGS: dict[str, list[str]] = {
    "cash": ["CashAndCashEquivalentsAtCarryingValue",
             "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
             "Cash"],
    "st_investments": ["ShortTermInvestments", "MarketableSecuritiesCurrent",
                       "AvailableForSaleSecuritiesDebtSecuritiesCurrent"],
    "receivables": ["AccountsReceivableNetCurrent", "ReceivablesNetCurrent"],
    "inventory": ["InventoryNet"],
    "current_assets": ["AssetsCurrent"],
    "lt_investments": ["MarketableSecuritiesNoncurrent", "LongTermInvestments",
                       "AvailableForSaleSecuritiesDebtSecuritiesNoncurrent"],
    "ppe_net": ["PropertyPlantAndEquipmentNet"],
    "total_assets": ["Assets"],
    "accounts_payable": ["AccountsPayableCurrent", "AccountsPayableAndAccruedLiabilitiesCurrent"],
    "current_liabilities": ["LiabilitiesCurrent"],
    "debt_current_total": ["DebtCurrent"],
    "ltd_current": ["LongTermDebtCurrent", "LongTermDebtAndCapitalLeaseObligationsCurrent"],
    "st_borrowings": ["CommercialPaper", "ShortTermBorrowings", "OtherShortTermBorrowings"],
    "ltd_noncurrent": ["LongTermDebtNoncurrent", "LongTermDebtAndCapitalLeaseObligations",
                       "LongTermNotesPayable"],
    "ltd_total": ["LongTermDebt"],
    "total_liabilities": ["Liabilities"],
    "equity": ["StockholdersEquity",
               "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "minority_interest": ["MinorityInterest"],
}

# ----------------------------------------------------------------------------
# IFRS equivalents (taxonomy "ifrs-full") for foreign companies filing a 20-F,
# e.g. ASML, SAP, Novo Nordisk, Shell, TotalEnergies. Values are in the company's
# reporting currency (usually EUR).
# ----------------------------------------------------------------------------
IFRS_FLOW_TAGS: dict[str, list[str]] = {
    "revenue": ["Revenue", "RevenueFromContractsWithCustomers", "RevenueFromSaleOfGoods"],
    "cogs": ["CostOfSales"],
    "gross_profit": ["GrossProfit"],
    "rnd": ["ResearchAndDevelopmentExpense"],
    "sga": ["SellingGeneralAndAdministrativeExpense"],
    "operating_income": ["ProfitLossFromOperatingActivities"],
    "interest_expense": ["FinanceCosts", "InterestExpense"],
    "pretax_income": ["ProfitLossBeforeTax"],
    "income_tax": ["IncomeTaxExpenseContinuingOperations"],
    "net_income": ["ProfitLossAttributableToOwnersOfParent", "ProfitLoss"],
    "da": ["DepreciationAndAmortisationExpense",
           "DepreciationAmortisationAndImpairmentLossReversalOfImpairmentLossRecognisedInProfitOrLoss",
           "AdjustmentsForDepreciationAndAmortisationExpense", "DepreciationExpense"],
    "sbc": ["AdjustmentsForSharebasedPayments", "ExpenseFromSharebasedPaymentTransactionsWithEmployees"],
    "cfo": ["CashFlowsFromUsedInOperatingActivities"],
    "capex": ["PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities",
              "PurchaseOfPropertyPlantAndEquipment"],
    "dividends": ["DividendsPaidClassifiedAsFinancingActivities", "DividendsPaid"],
    "buybacks": ["PaymentsToAcquireOrRedeemEntitysShares", "PurchaseOfTreasuryShares"],
}
IFRS_PER_SHARE_TAGS = {"eps_diluted": ["DilutedEarningsLossPerShare",
                                       "DilutedEarningsLossPerShareFromContinuingOperations"]}
IFRS_SHARE_FLOW_TAGS = {"diluted_shares": ["AdjustedWeightedAverageShares", "WeightedAverageShares"]}
IFRS_INSTANT_TAGS: dict[str, list[str]] = {
    "cash": ["CashAndCashEquivalents"],
    "st_investments": ["CurrentInvestments", "ShorttermDepositsNotClassifiedAsCashEquivalents",
                       "OtherCurrentFinancialAssets"],
    "receivables": ["TradeAndOtherCurrentReceivables", "CurrentTradeReceivables"],
    "inventory": ["Inventories"],
    "current_assets": ["CurrentAssets"],
    "lt_investments": ["NoncurrentInvestments", "OtherNoncurrentFinancialAssets"],
    "ppe_net": ["PropertyPlantAndEquipment"],
    "total_assets": ["Assets"],
    "accounts_payable": ["TradeAndOtherCurrentPayables", "TradeAndOtherCurrentPayablesToTradeSuppliers"],
    "current_liabilities": ["CurrentLiabilities"],
    "debt_current_total": ["CurrentBorrowingsAndCurrentPortionOfNoncurrentBorrowings"],
    "ltd_current": ["CurrentPortionOfLongtermBorrowings"],
    "st_borrowings": ["ShorttermBorrowings"],
    "ltd_noncurrent": ["NoncurrentPortionOfNoncurrentBorrowings", "LongtermBorrowings"],
    "ltd_total": [],
    "total_liabilities": ["Liabilities"],
    "equity": ["EquityAttributableToOwnersOfParent", "Equity"],
    "minority_interest": ["NoncontrollingInterests"],
}

TAG_SETS = {
    "us-gaap": (FLOW_TAGS, PER_SHARE_TAGS, SHARE_FLOW_TAGS, INSTANT_TAGS),
    "ifrs-full": (IFRS_FLOW_TAGS, IFRS_PER_SHARE_TAGS, IFRS_SHARE_FLOW_TAGS, IFRS_INSTANT_TAGS),
}


# Rows shown in each statement (after derivation), with display labels.
STATEMENT_LAYOUT = {
    "Income Statement": {
        "revenue": "Revenue", "cogs": "Cost of revenue", "gross_profit": "Gross profit",
        "rnd": "R&D", "sga": "SG&A", "ebit": "Operating income (EBIT)",
        "da": "D&A", "ebitda": "EBITDA", "interest_expense": "Interest expense",
        "pretax_income": "Pre-tax income", "income_tax": "Income tax",
        "net_income": "Net income", "eps_diluted": "Diluted EPS",
        "diluted_shares": "Diluted shares",
    },
    "Balance Sheet": {
        "cash": "Cash & equivalents", "st_investments": "Short-term investments",
        "receivables": "Receivables", "inventory": "Inventory",
        "current_assets": "Total current assets",
        "lt_investments": "Long-term investments", "ppe_net": "PP&E (net)",
        "total_assets": "Total assets", "accounts_payable": "Accounts payable",
        "current_liabilities": "Total current liabilities", "total_debt": "Total debt",
        "total_liabilities": "Total liabilities", "equity": "Shareholders' equity",
        "minority_interest": "Minority interest", "nwc": "Operating working capital",
    },
    "Cash Flow": {
        "cfo": "Cash from operations", "capex": "Capital expenditures",
        "fcf": "Free cash flow (CFO - CapEx)", "sbc": "Stock-based comp",
        "dividends": "Dividends paid", "buybacks": "Share buybacks",
    },
}


# ----------------------------------------------------------------------------
# Low-level fact extraction
# ----------------------------------------------------------------------------
def _facts_frame(facts: dict, tag: str, unit_pref: tuple[str, ...]) -> pd.DataFrame:
    """All reported values for one us-gaap tag as a DataFrame (one row per fact)."""
    node = facts.get("facts", {}).get("us-gaap", {}).get(tag)
    if not node:
        return pd.DataFrame()
    units = node.get("units", {})
    unit = next((u for u in unit_pref if u in units), None)
    if unit is None:
        return pd.DataFrame()
    df = pd.DataFrame(units[unit])
    if df.empty:
        return df
    df["end"] = pd.to_datetime(df["end"])
    df["start"] = pd.to_datetime(df["start"]) if "start" in df else pd.NaT
    df["filed"] = pd.to_datetime(df["filed"])
    df["days"] = (df["end"] - df["start"]).dt.days
    return df


def _latest_per_end(df: pd.DataFrame) -> pd.Series:
    df = df.sort_values("filed").drop_duplicates(subset="end", keep="last")
    return df.set_index("end")["val"].astype(float).sort_index()


def _annual_series(facts, tags, instant: bool, unit_pref) -> pd.Series:
    """Annual values keyed on period-end date, merging tags in priority order."""
    merged = pd.Series(dtype=float)
    for tag in tags:
        df = _facts_frame(facts, tag, unit_pref)
        if df.empty:
            continue
        df = df[df["form"].isin(ANNUAL_FORMS)]
        if instant:
            df = df[df["start"].isna()] if "start" in df else df
        else:
            df = df[df["days"].between(350, 380)]
        if df.empty:
            continue
        merged = merged.combine_first(_latest_per_end(df)) if len(merged) else _latest_per_end(df)
    return merged.sort_index()


def _ytd_pair(facts, tags, fy_end: pd.Timestamp, unit_pref, target_end=None):
    """(current YTD value, prior-year YTD value, current period end) from 10-Qs."""
    for tag in tags:
        df = _facts_frame(facts, tag, unit_pref)
        if df.empty or "start" not in df:
            continue
        df = df[df["form"].isin(QUARTERLY_FORMS) & df["start"].notna()]
        cur = df[(abs((df["start"] - (fy_end + pd.Timedelta(days=1))).dt.days) <= 7)
                 & (df["end"] > fy_end)]
        if target_end is not None:
            cur = cur[cur["end"] == target_end]
        if cur.empty:
            continue
        cur = cur.sort_values(["end", "filed"]).iloc[-1]
        prior_end = cur["end"] - pd.DateOffset(years=1)
        prior = df[(abs((df["end"] - prior_end).dt.days) <= 7)
                   & (abs(df["days"] - cur["days"]) <= 7)]
        if prior.empty:
            continue
        prior = prior.sort_values("filed").iloc[-1]
        return float(cur["val"]), float(prior["val"]), cur["end"]
    return None


def _latest_instant(facts, tags, end: pd.Timestamp, unit_pref):
    for tag in tags:
        df = _facts_frame(facts, tag, unit_pref)
        if df.empty:
            continue
        df = df[(df["end"] == end) & (df["start"].isna() if "start" in df else True)]
        if not df.empty:
            return float(df.sort_values("filed")["val"].iloc[-1])
    return np.nan


def _shares_outstanding(facts) -> tuple[float, str | None]:
    """Latest cover-page share count (sums share classes reported in one filing)."""
    node = facts.get("facts", {}).get("dei", {}).get("EntityCommonStockSharesOutstanding")
    if not node:
        return np.nan, None
    df = pd.DataFrame(node["units"].get("shares", []))
    if df.empty:
        return np.nan, None
    df["end"] = pd.to_datetime(df["end"])
    df["filed"] = pd.to_datetime(df["filed"])
    latest = df.sort_values(["filed", "end"]).iloc[-1]
    same = df[(df["accn"] == latest["accn"]) & (df["end"] == latest["end"])]
    return float(same["val"].drop_duplicates().sum()), latest["end"].date().isoformat()


# ----------------------------------------------------------------------------
# Derived metrics
# ----------------------------------------------------------------------------
def _derive(df: pd.DataFrame) -> pd.DataFrame:
    """Add derived line items. Columns = periods, rows = line items."""
    def row(name):
        return df.loc[name] if name in df.index else pd.Series(np.nan, index=df.columns)

    out = df.copy()
    rev, cogs = row("revenue"), row("cogs")
    out.loc["gross_profit"] = row("gross_profit").fillna(rev - cogs)
    out.loc["cogs"] = cogs.fillna(rev - out.loc["gross_profit"])
    out.loc["ebit"] = row("operating_income").fillna(
        row("pretax_income") + row("interest_expense").fillna(0))
    out.loc["ebitda"] = out.loc["ebit"] + row("da")

    # Debt: DebtCurrent already includes short-term borrowings; otherwise add up.
    current_debt = row("debt_current_total").fillna(
        row("ltd_current").fillna(0) + row("st_borrowings").fillna(0))
    noncurrent = row("ltd_noncurrent")
    # If only LongTermDebt (total, incl. current portion) exists, avoid double counting.
    noncurrent = noncurrent.fillna(row("ltd_total") - row("ltd_current").fillna(0))
    out.loc["total_debt"] = current_debt.fillna(0) + noncurrent.fillna(0)

    # Operating (non-cash, non-debt) working capital.
    cash_like = row("cash").fillna(0) + row("st_investments").fillna(0)
    nwc_full = (row("current_assets") - cash_like) - (row("current_liabilities") - current_debt.fillna(0))
    nwc_simple = (row("receivables").fillna(0) + row("inventory").fillna(0)
                  - row("accounts_payable").fillna(0))
    nwc_simple = nwc_simple.where(row("receivables").notna() | row("accounts_payable").notna())
    out.loc["nwc"] = nwc_full.fillna(nwc_simple)

    out.loc["capex"] = row("capex").abs()
    out.loc["fcf"] = row("cfo") - out.loc["capex"]
    pretax = row("pretax_income")
    out.loc["tax_rate"] = (row("income_tax") / pretax).where(pretax > 0)
    return out


# ----------------------------------------------------------------------------
# Public API
# ----------------------------------------------------------------------------
@dataclass
class Financials:
    ticker: str
    name: str
    cik: int
    sic: str | None
    sic_description: str | None
    annual: pd.DataFrame                  # rows = line items, cols = fiscal-year labels
    ttm: pd.Series | None                 # latest twelve months (if a newer 10-Q exists)
    ttm_end: str | None
    shares_outstanding: float
    shares_date: str | None
    latest_10k: Filing | None             # latest annual report (10-K, or 20-F for foreign filers)
    warnings: list[str] = field(default_factory=list)
    currency: str = "USD"                 # reporting currency of the statements
    source: str = "SEC EDGAR"
    taxonomy: str = "us-gaap"             # "us-gaap", "ifrs-full" or "yahoo"
    sector: str | None = None             # Yahoo sector/industry (used when there is no SIC code)
    industry: str | None = None

    @property
    def is_financial(self) -> bool:
        try:
            return 6000 <= int(self.sic) <= 6799
        except (TypeError, ValueError):
            return False

    def statement(self, name: str, include_ttm: bool = True) -> pd.DataFrame:
        """Formatted statement in millions of the reporting currency (EPS per share)."""
        layout = STATEMENT_LAYOUT[name]
        data = self.annual.copy()
        if include_ttm and self.ttm is not None:
            data[f"TTM {self.ttm_end}"] = self.ttm
        data = data.reindex(list(layout))
        scale = pd.Series(1e6, index=data.index)
        if "eps_diluted" in scale.index:
            scale["eps_diluted"] = 1.0
        table = data.div(scale, axis=0)
        table.index = [layout[k] for k in table.index]
        return table.dropna(how="all")


def _detect_taxonomy_and_currency(facts: dict) -> tuple[str, str]:
    """US-GAAP if present, else IFRS; currency = the unit most used by revenue/asset facts."""
    all_facts = facts.get("facts", {})
    for tax in ("us-gaap", "ifrs-full"):
        node = all_facts.get(tax)
        if not node:
            continue
        flows, _, _, instants = TAG_SETS[tax]
        counts: dict[str, int] = {}
        for tag in flows["revenue"] + instants["total_assets"] + flows["net_income"]:
            for unit, rows in node.get(tag, {}).get("units", {}).items():
                if "/" not in unit and unit != "shares":
                    counts[unit] = counts.get(unit, 0) + len(rows)
        if counts:
            return tax, max(counts, key=counts.get)
    raise ValueError("No US-GAAP or IFRS financial data in this company's SEC filings.")


def load_financials(client: SECClient, ticker: str, years: int = 5,
                    include_ttm: bool = True) -> Financials:
    info = client.lookup(ticker)
    cik = info["cik"]
    facts = client.company_facts(cik)
    try:
        subs = client.submissions(cik)
        sic, sic_desc = subs.get("sic"), subs.get("sicDescription")
        latest_10k = client.latest_filing(cik)
    except Exception:  # submissions endpoint is optional for the model
        sic = sic_desc = latest_10k = None

    taxonomy, ccy = _detect_taxonomy_and_currency(facts)
    flow_tags, ps_tags, sh_tags, inst_tags = TAG_SETS[taxonomy]
    # The helpers read the "us-gaap" slot; point it at whichever taxonomy the company uses.
    view = {"facts": {"us-gaap": facts["facts"][taxonomy], "dei": facts["facts"].get("dei", {})}}
    money, per_share = (ccy,), (f"{ccy}/shares",)

    warnings: list[str] = []
    if taxonomy == "ifrs-full":
        warnings.append(f"Foreign filer: IFRS statements from the annual report (20-F), in {ccy}.")
    series: dict[str, pd.Series] = {}
    for key, tags in flow_tags.items():
        series[key] = _annual_series(view, tags, False, money)
    for key, tags in ps_tags.items():
        series[key] = _annual_series(view, tags, False, per_share)
    for key, tags in sh_tags.items():
        series[key] = _annual_series(view, tags, False, ("shares",))
    for key, tags in inst_tags.items():
        series[key] = _annual_series(view, tags, True, money)

    # Fiscal year ends = periods where we have revenue (or net income as fallback).
    anchor = series["revenue"].dropna()
    if anchor.empty:
        anchor = series["net_income"].dropna()
        warnings.append("No revenue tag found - revenue-based projections will not work.")
    if anchor.empty:
        raise ValueError(f"No annual financial data found for {ticker} in SEC filings.")
    fy_ends = anchor.index[-years:]

    annual = pd.DataFrame({k: s.reindex(fy_ends) for k, s in series.items()}).T
    annual = _derive(annual)
    annual.columns = [f"FY{d.year}" for d in fy_ends]
    last_fy_end = fy_ends[-1]

    # ---- Trailing twelve months from the latest 10-Q (US filers only) -------
    ttm, ttm_end = None, None
    if include_ttm:
        rev_tags = flow_tags["revenue"] if not series["revenue"].dropna().empty else flow_tags["net_income"]
        anchor_pair = _ytd_pair(view, rev_tags, last_fy_end, money)
        if anchor_pair:
            end = anchor_pair[2]
            values = {}
            for group, unit in [(flow_tags, money), (sh_tags, ("shares",))]:
                for key, tags in group.items():
                    pair = _ytd_pair(view, tags, last_fy_end, unit, target_end=end)
                    fy_val = series[key].get(last_fy_end, np.nan)
                    if key == "diluted_shares":   # a weighted share count is not additive
                        values[key] = pair[0] if pair else fy_val
                    else:
                        values[key] = fy_val + pair[0] - pair[1] if pair else np.nan
            for key, tags in inst_tags.items():
                values[key] = _latest_instant(view, tags, end, money)
            ttm_df = _derive(pd.DataFrame({"ttm": values}))
            ttm = ttm_df["ttm"]
            ttm_end = end.date().isoformat()

    shares, shares_date = _shares_outstanding(view)
    diluted = annual.loc["diluted_shares"].dropna()
    if diluted.size:
        ref = diluted.iloc[-1]
        if not np.isfinite(shares) or abs(shares / ref - 1) > 0.30:
            warnings.append(
                "Cover-page share count missing or inconsistent with diluted shares; "
                "using latest diluted weighted-average shares instead.")
            shares, shares_date = ref, annual.columns[-1]

    fin = Financials(
        ticker=info["ticker"], name=facts.get("entityName", info["name"]), cik=cik,
        sic=sic, sic_description=sic_desc, annual=annual, ttm=ttm, ttm_end=ttm_end,
        shares_outstanding=shares, shares_date=shares_date, latest_10k=latest_10k,
        warnings=warnings, currency=ccy, source="SEC EDGAR", taxonomy=taxonomy,
    )
    if fin.is_financial:
        fin.warnings.append(
            f"SIC {sic} ({sic_desc}) is a financial company. An unlevered FCFF DCF is not "
            "meaningful for banks/insurers - treat results with caution.")
    return fin


In [ ]:
%%writefile dcf/market.py
"""
Market inputs for the cost of capital: share price, beta and the risk-free rate.

Data comes from Yahoo Finance via ``yfinance``. Beta is *computed*, not scraped:
an OLS regression of 5 years of monthly stock returns on the S&P 500, then
Blume-adjusted (0.67 x raw + 0.33) toward the market average of 1.0.
Every value can be overridden by hand if Yahoo is unavailable.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
import pandas as pd

DEFAULT_RISK_FREE = 0.0425   # fallback only; the live 10Y Treasury yield is used when available
BENCHMARK = "^GSPC"          # S&P 500
TEN_YEAR = "^TNX"            # CBOE 10-year Treasury yield index

# Market index used for the beta regression, by trading currency.
BENCHMARKS = {"USD": "^GSPC", "EUR": "^STOXX50E", "GBP": "^FTSE", "CHF": "^SSMI",
              "SEK": "^STOXX", "DKK": "^STOXX", "NOK": "^STOXX", "PLN": "^STOXX",
              "JPY": "^N225", "CAD": "^GSPTSE", "HKD": "^HSI", "AUD": "^AXJO"}
# Quotes in minor units (London pence, Israeli agorot, SA cents).
MINOR_UNITS = {"GBp": ("GBP", 100), "GBX": ("GBP", 100), "ILA": ("ILS", 100), "ZAc": ("ZAR", 100)}


@dataclass
class MarketData:
    price: float                     # share price in the REPORTING currency of the statements
    beta: float
    raw_beta: float | None
    risk_free: float
    price_date: str | None = None
    notes: list[str] = field(default_factory=list)
    currency: str = "USD"            # reporting currency (what price is expressed in)
    price_currency: str = "USD"      # currency the stock trades in
    price_original: float | None = None
    fx_rate: float = 1.0             # 1 unit of price_currency = fx_rate units of reporting currency


def normalise_currency(price: float, ccy: str) -> tuple[float, str]:
    """Convert minor-unit quotes (e.g. London pence 'GBp') into the major currency."""
    if ccy in MINOR_UNITS:
        major, factor = MINOR_UNITS[ccy]
        return price / factor, major
    return price, ccy


def fx_rate(from_ccy: str, to_ccy: str, lookup=None) -> float:
    """Units of to_ccy per 1 from_ccy. `lookup(pair)` returns a quote for e.g. 'USDEUR=X'."""
    if from_ccy == to_ccy:
        return 1.0
    if lookup is None:
        import yfinance as yf

        def lookup(pair):
            return float(yf.Ticker(pair).history(period="5d")["Close"].dropna().iloc[-1])
    return float(lookup(f"{from_ccy}{to_ccy}=X"))


def regression_beta(stock: pd.Series, market: pd.Series) -> tuple[float, float, int]:
    """Raw OLS beta, R-squared and number of observations from two return series."""
    df = pd.concat([stock, market], axis=1).dropna()
    if len(df) < 24:
        raise ValueError("Need at least 24 monthly observations for a beta.")
    x, y = df.iloc[:, 1].values, df.iloc[:, 0].values
    beta = np.cov(y, x, ddof=1)[0, 1] / np.var(x, ddof=1)
    r2 = np.corrcoef(x, y)[0, 1] ** 2
    return float(beta), float(r2), len(df)


def blume_adjust(raw_beta: float) -> float:
    return 0.67 * raw_beta + 0.33


def fetch_consensus_growth(ticker: str) -> tuple[float | None, str]:
    """Analysts' average revenue growth for this and next fiscal year (Yahoo Finance)."""
    try:
        import yfinance as yf
        est = yf.Ticker(ticker).revenue_estimate
        growth = est.loc[[i for i in ("0y", "+1y") if i in est.index], "growth"].dropna().astype(float)
        if growth.empty:
            return None, "no analyst estimates"
        return float(growth.mean()), f"analyst consensus revenue growth ({', '.join(f'{g:.1%}' for g in growth)})"
    except Exception as exc:  # noqa: BLE001
        return None, f"analyst estimates unavailable ({type(exc).__name__})"


def fetch_market_data(ticker: str, price: float | None = None, beta: float | None = None,
                      risk_free: float | None = None, years: int = 5,
                      reporting_currency: str = "USD", price_currency: str | None = None) -> MarketData:
    """Fetch price, beta and risk-free rate; any argument you pass overrides the live value.

    The price is converted into the statements' reporting currency (e.g. ASML's US listing
    trades in USD but reports in EUR), so value per share and price are comparable.
    A price you pass yourself is taken to be in `price_currency` (default: reporting currency).
    """
    notes: list[str] = []
    raw_beta = None
    price_date = None
    manual_price = price is not None
    price_ccy = price_currency or (reporting_currency if manual_price else None)
    try:
        import yfinance as yf
    except ImportError:  # pragma: no cover
        yf = None
        notes.append("yfinance not installed - using manual/default market inputs.")

    if yf is not None:
        tk = yf.Ticker(ticker)
        # ---- price -----------------------------------------------------------
        if price is None:
            try:
                hist = tk.history(period="5d", auto_adjust=False)
                price = float(hist["Close"].dropna().iloc[-1])
                price_date = hist.index[-1].date().isoformat()
            except Exception as exc:  # noqa: BLE001
                notes.append(f"Could not fetch price ({exc}).")
        if price_ccy is None:
            try:
                price_ccy = tk.fast_info["currency"]
            except Exception:  # noqa: BLE001
                price_ccy = reporting_currency
                notes.append(f"Trading currency unknown - assumed {reporting_currency}.")
        # ---- beta ------------------------------------------------------------
        if beta is None:
            bench = BENCHMARKS.get(normalise_currency(1.0, price_ccy or "USD")[1], BENCHMARK)
            try:
                px = yf.download([ticker, bench], period=f"{years}y", interval="1mo",
                                 auto_adjust=True, progress=False)["Close"]
                rets = px.pct_change(fill_method=None).dropna()
                raw_beta, r2, n = regression_beta(rets[ticker], rets[bench])
                beta = blume_adjust(raw_beta)
                notes.append(f"Beta: {n} monthly returns vs {bench}, raw {raw_beta:.2f} "
                             f"(R² {r2:.2f}), Blume-adjusted {beta:.2f}.")
            except Exception as exc:  # noqa: BLE001
                notes.append(f"Could not compute beta ({exc}); using 1.0.")
        # ---- risk-free rate --------------------------------------------------
        if risk_free is None:
            try:
                tnx = float(yf.Ticker(TEN_YEAR).history(period="5d")["Close"].dropna().iloc[-1])
                risk_free = tnx / 100 if tnx < 20 else tnx / 1000
                notes.append(f"Risk-free rate: 10Y US Treasury yield {risk_free:.2%}.")
            except Exception as exc:  # noqa: BLE001
                notes.append(f"Could not fetch 10Y yield ({exc}).")

    if price is None:
        raise ValueError("No share price available - pass price=... manually.")
    price_ccy = price_ccy or reporting_currency

    # ---- currency alignment --------------------------------------------------
    original = price
    price, major = normalise_currency(price, price_ccy)
    if major != price_ccy:
        notes.append(f"Price quoted in {price_ccy}; converted to {major}.")
    rate = 1.0
    if major != reporting_currency:
        rate = fx_rate(major, reporting_currency)
        price = price * rate
        notes.append(f"Price {original:,.2f} {major} converted at {rate:.4f} = {price:,.2f} "
                     f"{reporting_currency} (statements are in {reporting_currency}). If this is an ADR "
                     "where one ADR is not one ordinary share (e.g. BP, Shell), use the home-exchange "
                     "ticker instead (e.g. SHEL.L).")
    if beta is None:
        beta = 1.0
    if risk_free is None:
        risk_free = DEFAULT_RISK_FREE
        notes.append(f"Risk-free rate defaulted to {risk_free:.2%}.")
    if reporting_currency != "USD" and "Risk-free rate:" in " ".join(notes):
        notes.append(f"Cash flows are in {reporting_currency} but the risk-free rate is the US 10Y yield. "
                     "Use the government bond yield of the same currency (e.g. German 10Y Bund for EUR) "
                     "via RISK_FREE for a consistent discount rate.")
    return MarketData(price=price, beta=beta, raw_beta=raw_beta, risk_free=risk_free,
                      price_date=price_date, notes=notes, currency=reporting_currency,
                      price_currency=major, price_original=original, fx_rate=rate)


In [ ]:
%%writefile dcf/model.py
"""
Unlevered free-cash-flow (FCFF) DCF.

    FCFF  = EBIT x (1 - tax) + D&A - CapEx - change in NWC
    WACC  = E/(D+E) x Ke + D/(D+E) x Kd x (1 - tax),   Ke = rf + beta x ERP   (CAPM)
    TV    = FCFF_N x (1 + g) / (WACC - g)                                    (Gordon growth)
    EV    = sum PV(FCFF) + PV(TV)
    Equity = EV - debt + cash & short-term investments - minority interest

Default assumptions are derived from the company's own history (3-year averages,
clipped to sensible ranges) and every one of them can be overridden.
"""
from __future__ import annotations

from dataclasses import dataclass, field, replace

import numpy as np
import pandas as pd

from .financials import Financials
from .market import MarketData

DEFAULT_ERP = 0.045          # equity risk premium - update from Damodaran's monthly implied ERP
DEFAULT_TERMINAL_GROWTH = 0.025
MARGINAL_TAX = 0.21          # US federal statutory rate (fallback)
CURRENCY_SYMBOLS = {"USD": "$", "EUR": "€", "GBP": "£", "JPY": "¥", "CHF": "CHF ", "SEK": "SEK ",
                    "DKK": "DKK ", "NOK": "NOK ", "CAD": "C$", "AUD": "A$", "HKD": "HK$"}


def money(x: float, currency: str = "USD", decimals: int = 2) -> str:
    sym = CURRENCY_SYMBOLS.get(currency, currency + " ")
    return f"{sym}{x:,.{decimals}f}"


@dataclass
class Assumptions:
    projection_years: int = 5
    initial_growth: float = 0.05          # year-1 revenue growth, fades linearly to terminal_growth
    terminal_growth: float = DEFAULT_TERMINAL_GROWTH
    revenue_growth: list[float] | None = None   # explicit path overrides the fade
    ebit_margin: float = 0.15             # base-year EBIT margin
    target_ebit_margin: float | None = None  # if set, margin moves linearly to this by the final year
    tax_rate: float = MARGINAL_TAX
    da_pct: float = 0.03                  # D&A as % of revenue
    capex_pct: float = 0.04               # CapEx as % of revenue
    nwc_pct: float = 0.05                 # operating working capital as % of revenue
    risk_free: float = 0.0425
    beta: float = 1.0
    erp: float = DEFAULT_ERP
    pre_tax_cost_of_debt: float = 0.055
    wacc_override: float | None = None
    mid_year: bool = True                 # mid-year discounting convention
    include_lt_investments: bool = False  # count long-term marketable securities as cash-like
    sources: dict = field(default_factory=dict, repr=False)   # where each default came from

    def growth_path(self) -> np.ndarray:
        n = self.projection_years
        if self.revenue_growth is not None:
            path = list(self.revenue_growth)[:n]
            return np.array(path + [path[-1]] * (n - len(path)))
        # n+1 evenly spaced points from year-1 growth to terminal growth; the last
        # point (= terminal growth) belongs to the perpetuity, not the forecast.
        return np.linspace(self.initial_growth, self.terminal_growth, n + 1)[:-1]

    def margin_path(self) -> np.ndarray:
        n = self.projection_years
        target = self.ebit_margin if self.target_ebit_margin is None else self.target_ebit_margin
        return np.linspace(self.ebit_margin, target, n + 1)[1:]


@dataclass
class DCFResult:
    company: str
    ticker: str
    base_label: str
    projection: pd.DataFrame
    wacc: dict
    enterprise_value: float
    pv_fcff: float
    pv_terminal: float
    terminal_value: float
    bridge: dict
    equity_value: float
    shares: float
    value_per_share: float
    price: float
    upside: float
    implied_exit_multiple: float
    assumptions: "Assumptions"             # or growth.GrowthAssumptions
    warnings: list[str] = field(default_factory=list)
    model: str = "standard"                # "standard" or "growth"
    currency: str = "USD"                  # reporting currency of values and price

    @property
    def terminal_share_of_ev(self) -> float:
        return self.pv_terminal / self.enterprise_value

    def summary(self) -> pd.Series:
        return pd.Series({
            "Model": ("Growth-stage DCF (10-yr, 2 stages)" if self.model == "growth"
                      else "Standard DCF (5-yr)"),
            "WACC": (f"{self.wacc['wacc']:.2%} -> {self.wacc['terminal_wacc']:.2%} (mature)"
                     if "terminal_wacc" in self.wacc else f"{self.wacc['wacc']:.2%}"),
            "Terminal growth": f"{self.assumptions.terminal_growth:.2%}",
            f"PV of FCFF ({self.currency}m)": f"{self.pv_fcff / 1e6:,.0f}",
            f"PV of terminal value ({self.currency}m)": f"{self.pv_terminal / 1e6:,.0f}",
            f"Enterprise value ({self.currency}m)": f"{self.enterprise_value / 1e6:,.0f}",
            f"Equity value ({self.currency}m)": f"{self.equity_value / 1e6:,.0f}",
            "Shares (m)": f"{self.shares / 1e6:,.1f}",
            "Intrinsic value / share": money(self.value_per_share, self.currency),
            "Current price": money(self.price, self.currency),
            "Upside / (downside)": f"{self.upside:+.1%}",
            "Terminal value % of EV": f"{self.terminal_share_of_ev:.0%}",
            "Implied exit EV/EBITDA": f"{self.implied_exit_multiple:.1f}x",
        }, name=self.ticker)


# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------
def _clip(x, lo, hi, fallback):
    if x is None or not np.isfinite(x):
        return fallback
    return float(min(max(x, lo), hi))


def base_period(fin: Financials, use_ttm: bool = True) -> tuple[pd.Series, str]:
    """The statement column the projections start from (TTM if available)."""
    if use_ttm and fin.ttm is not None and np.isfinite(fin.ttm.get("revenue", np.nan)):
        return fin.ttm, f"TTM to {fin.ttm_end}"
    return fin.annual.iloc[:, -1], fin.annual.columns[-1]


def _ratio_avg(fin: Financials, num: str, den: str = "revenue", n: int = 3) -> float:
    a = fin.annual
    if num not in a.index or den not in a.index:
        return np.nan
    r = (a.loc[num] / a.loc[den]).replace([np.inf, -np.inf], np.nan).dropna()
    return float(r.iloc[-n:].mean()) if len(r) else np.nan


def default_assumptions(fin: Financials, mkt: MarketData, use_ttm: bool = True,
                        **overrides) -> Assumptions:
    """Derive every assumption from the company's own history, then apply overrides."""
    a = fin.annual
    base, _ = base_period(fin, use_ttm)
    src: dict[str, str] = {}

    rev = a.loc["revenue"].dropna()
    if len(rev) >= 2:
        k = min(3, len(rev) - 1)
        cagr = (rev.iloc[-1] / rev.iloc[-1 - k]) ** (1 / k) - 1
        src["initial_growth"] = f"{k}-yr revenue CAGR {cagr:.1%} (clipped to -5%..25%)"
    else:
        cagr = np.nan
        src["initial_growth"] = "default (insufficient history)"
    growth = _clip(cagr, -0.05, 0.25, 0.05)

    margin = _clip(_ratio_avg(fin, "ebit"), -0.20, 0.60, 0.10)
    src["ebit_margin"] = "3-yr average EBIT margin"

    tax = _clip(a.loc["tax_rate"].dropna().iloc[-3:].mean() if a.loc["tax_rate"].notna().any()
                else np.nan, 0.10, 0.30, MARGINAL_TAX)
    src["tax_rate"] = "3-yr average effective tax rate (clipped to 10%..30%)"

    da = _clip(_ratio_avg(fin, "da"), 0.0, 0.25, 0.03)
    src["da_pct"] = "3-yr average D&A / revenue"
    capex = _clip(_ratio_avg(fin, "capex"), 0.0, 0.30, 0.04)
    src["capex_pct"] = "3-yr average CapEx / revenue"
    nwc_ratio = base.get("nwc", np.nan) / base.get("revenue", np.nan)
    src["nwc_pct"] = "base-period operating working capital / revenue"
    if not np.isfinite(nwc_ratio):
        nwc_ratio = a.loc["nwc"].iloc[-1] / a.loc["revenue"].iloc[-1]
        src["nwc_pct"] = "last fiscal year operating working capital / revenue"
    nwc = _clip(nwc_ratio, -0.15, 0.40, 0.05)

    # Cost of debt = interest expense / average debt, bounded around the risk-free rate.
    rf = mkt.risk_free
    debt = a.loc["total_debt"].replace(0, np.nan).dropna()
    interest = a.loc["interest_expense"].dropna()
    kd = np.nan
    if len(debt) and len(interest):
        avg_debt = debt.iloc[-2:].mean()
        kd = interest.iloc[-1] / avg_debt
        src["pre_tax_cost_of_debt"] = f"interest expense / average debt = {kd:.2%} (bounded rf+0.5%..rf+6%)"
    else:
        src["pre_tax_cost_of_debt"] = "risk-free + 1.5% (no interest expense reported)"
    kd = _clip(kd, rf + 0.005, rf + 0.06, rf + 0.015)

    src["risk_free"] = "10Y US Treasury yield"
    src["beta"] = "5-yr monthly regression vs S&P 500, Blume-adjusted" if mkt.raw_beta else "manual / default"
    src["erp"] = "Default; update from Damodaran's implied ERP"
    src["terminal_growth"] = "~long-run nominal GDP / inflation proxy"

    assumptions = Assumptions(
        initial_growth=growth, ebit_margin=margin, tax_rate=tax, da_pct=da,
        capex_pct=capex, nwc_pct=nwc, risk_free=rf, beta=mkt.beta,
        pre_tax_cost_of_debt=kd, sources=src,
    )
    for k, v in overrides.items():
        if k in src and getattr(assumptions, k) != v:
            src[k] = "manual override"
    return replace(assumptions, **overrides)


def compute_wacc(fin: Financials, mkt: MarketData, a: Assumptions, base: pd.Series) -> dict:
    equity = mkt.price * fin.shares_outstanding
    debt = float(np.nan_to_num(base.get("total_debt", 0.0)))
    ke = a.risk_free + a.beta * a.erp
    kd_after = a.pre_tax_cost_of_debt * (1 - a.tax_rate)
    we = equity / (equity + debt)
    wacc = we * ke + (1 - we) * kd_after
    if a.wacc_override is not None:
        wacc = a.wacc_override
    return {"market_cap": equity, "debt": debt, "cost_of_equity": ke,
            "pre_tax_cost_of_debt": a.pre_tax_cost_of_debt, "after_tax_cost_of_debt": kd_after,
            "weight_equity": we, "weight_debt": 1 - we, "wacc": wacc}


# ----------------------------------------------------------------------------
# The model
# ----------------------------------------------------------------------------
def run_dcf(fin: Financials, mkt: MarketData, a: Assumptions, use_ttm: bool = True) -> DCFResult:
    base, base_label = base_period(fin, use_ttm)
    warnings = list(fin.warnings)
    w = compute_wacc(fin, mkt, a, base)
    r, g, n = w["wacc"], a.terminal_growth, a.projection_years
    if r <= g:
        raise ValueError(f"WACC ({r:.2%}) must exceed terminal growth ({g:.2%}).")

    growth, margin = a.growth_path(), a.margin_path()
    revenue = base["revenue"] * np.cumprod(1 + growth)
    prev_rev = np.concatenate([[base["revenue"]], revenue[:-1]])
    ebit = revenue * margin
    taxes = ebit.clip(min=0) * a.tax_rate
    nopat = ebit - taxes
    da = revenue * a.da_pct
    capex = revenue * a.capex_pct
    d_nwc = (revenue - prev_rev) * a.nwc_pct
    fcff = nopat + da - capex - d_nwc

    t = np.arange(1, n + 1) - (0.5 if a.mid_year else 0.0)
    df = 1 / (1 + r) ** t
    pv = fcff * df

    tv = fcff[-1] * (1 + g) / (r - g)
    pv_tv = tv / (1 + r) ** n            # terminal value discounted from the end of year N
    ev = pv.sum() + pv_tv

    cash = float(np.nan_to_num(base.get("cash", 0))) + float(np.nan_to_num(base.get("st_investments", 0)))
    lt_inv = float(np.nan_to_num(base.get("lt_investments", 0))) if a.include_lt_investments else 0.0
    debt = w["debt"]
    minority = float(np.nan_to_num(base.get("minority_interest", 0)))
    equity = ev - debt + cash + lt_inv - minority
    per_share = equity / fin.shares_outstanding

    ebitda_n = ebit[-1] + da[-1]
    years = [f"Y{i}" for i in range(1, n + 1)]
    projection = pd.DataFrame({
        "Revenue": revenue, "Growth": growth, "EBIT": ebit, "EBIT margin": margin,
        "Taxes": -taxes, "NOPAT": nopat, "D&A": da, "CapEx": -capex,
        "Change in NWC": -d_nwc, "FCFF": fcff, "Discount factor": df, "PV of FCFF": pv,
    }, index=years).T

    if a.capex_pct < a.da_pct and g > 0:
        warnings.append("CapEx < D&A in the terminal year: growth without reinvestment "
                        "may overstate the terminal value.")
    if equity <= 0 or fcff[-1] <= 0:
        warnings.append("Negative free cash flow or equity value: a DCF on current margins does "
                        "not work here - set a target_ebit_margin for a normalised business.")
    if pv_tv / ev > 0.85:
        warnings.append(f"Terminal value is {pv_tv / ev:.0%} of EV - result is very "
                        "sensitive to WACC and terminal growth.")

    return DCFResult(
        company=fin.name, ticker=fin.ticker, base_label=base_label, projection=projection,
        wacc=w, enterprise_value=ev, pv_fcff=pv.sum(), pv_terminal=pv_tv, terminal_value=tv,
        bridge={"Enterprise value": ev, "Less: debt": -debt,
                "Plus: cash & ST investments": cash, "Plus: LT investments": lt_inv,
                "Less: minority interest": -minority, "Equity value": equity},
        equity_value=equity, shares=fin.shares_outstanding, value_per_share=per_share,
        price=mkt.price, upside=per_share / mkt.price - 1,
        implied_exit_multiple=tv / ebitda_n if ebitda_n > 0 else np.nan,
        assumptions=a, warnings=warnings, currency=fin.currency,
    )


def sensitivity(fin: Financials, mkt: MarketData, a: Assumptions,
                row: str, row_values, col: str, col_values, use_ttm: bool = True) -> pd.DataFrame:
    """Value per share across a grid of two assumptions (e.g. 'wacc_override' x 'terminal_growth')."""
    out = pd.DataFrame(index=list(row_values), columns=list(col_values), dtype=float)
    for rv in row_values:
        for cv in col_values:
            try:
                res = run_dcf(fin, mkt, replace(a, **{row: rv, col: cv}), use_ttm)
                out.loc[rv, cv] = res.value_per_share
            except ValueError:
                out.loc[rv, cv] = np.nan
    out.index.name, out.columns.name = row, col
    return out


def wacc_growth_grid(fin, mkt, a, result: DCFResult, step_w=0.005, step_g=0.005, n=2, use_ttm=True):
    """Classic WACC x terminal-growth table centred on the base case."""
    w0, g0 = result.wacc["wacc"], a.terminal_growth
    waccs = [round(w0 + i * step_w, 4) for i in range(-n, n + 1)]
    gs = [round(g0 + i * step_g, 4) for i in range(-n, n + 1)]
    return sensitivity(fin, mkt, a, "wacc_override", waccs, "terminal_growth", gs, use_ttm)


def scenarios(fin, mkt, a: Assumptions, use_ttm=True) -> pd.DataFrame:
    """Bear / base / bull on growth and margin."""
    cases = {
        "Bear": replace(a, initial_growth=a.initial_growth - 0.03,
                        target_ebit_margin=a.ebit_margin - 0.03),
        "Base": a,
        "Bull": replace(a, initial_growth=a.initial_growth + 0.03,
                        target_ebit_margin=a.ebit_margin + 0.03),
    }
    rows = {}
    for name, case in cases.items():
        res = run_dcf(fin, mkt, case, use_ttm)
        rows[name] = {"Y1 growth": case.initial_growth,
                      "Final EBIT margin": case.margin_path()[-1],
                      "Value / share": res.value_per_share, "Upside": res.upside}
    return pd.DataFrame(rows).T


In [ ]:
%%writefile dcf/classify.py
"""
Decide which valuation model fits a company.

Signals
-------
1. SIC industry code from EDGAR (computers, semiconductors, software, ...)
2. R&D intensity (R&D / revenue) - catches tech companies with "non-tech" SIC codes
3. Revenue growth (3-year CAGR and latest year)
4. AI score - how often the latest 10-K talks about AI, per 10,000 words

Routing
-------
* AI / hyper-growth tech  -> growth-stage DCF (long high-growth period)
* Growth tech             -> growth-stage DCF (shorter high-growth period)
* Mature tech, Traditional, Financial -> standard 5-year DCF

The thresholds are judgement calls and live in one place (THRESHOLDS) so they are
easy to tune. Running a few known names (e.g. NVDA, MSFT, KO) is a good calibration check.
"""
from __future__ import annotations

import html
import re
from dataclasses import dataclass, field

import numpy as np

from .financials import Financials

# SIC ranges treated as technology.
TECH_SIC_RANGES = [
    (3570, 3579, "Computer & office equipment"),
    (3660, 3679, "Communications equipment, semiconductors & electronic components"),
    (3820, 3829, "Measuring & control instruments"),
    (7370, 7379, "Software, internet & IT services"),
]

# Used when there is no SIC code (companies loaded from Yahoo Finance).
TECH_INDUSTRY_WORDS = ("semiconductor", "software", "internet", "information technology",
                       "computer", "electronic component", "communication equipment",
                       "scientific & technical instruments", "consumer electronics")

THRESHOLDS = {
    "rnd_intensity_tech": 0.08,   # R&D >= 8% of revenue counts as tech
    "ai_score_driven": 20.0,      # AI mentions per 10,000 words -> "AI-driven"
    "ai_score_exposed": 5.0,      # -> "AI-exposed"
    "hyper_growth": 0.30,         # 3-yr revenue CAGR
    "growth": 0.10,
}

AI_PATTERNS = [
    re.compile(r"\bartificial intelligence\b", re.I),
    re.compile(r"\bmachine learning\b", re.I),
    re.compile(r"\bdeep learning\b", re.I),
    re.compile(r"\bneural networks?\b", re.I),
    re.compile(r"\blarge language models?\b", re.I),
    re.compile(r"\bLLMs?\b"),
    re.compile(r"\bgenerative\b", re.I),
    re.compile(r"\bAI\b"),            # case-sensitive on purpose
]


@dataclass
class Classification:
    profile: str                 # e.g. "AI / hyper-growth tech"
    model: str                   # "growth" or "standard"
    is_tech: bool
    tech_reason: str | None
    rnd_intensity: float
    revenue_cagr: float
    latest_growth: float
    ai_score: float | None       # mentions per 10,000 words in the latest 10-K
    ai_mentions: int | None
    reasons: list[str] = field(default_factory=list)

    def summary(self) -> str:
        lines = [f"Profile: **{self.profile}** -> {self.model} DCF"]
        lines += [f"- {r}" for r in self.reasons]
        return "\n".join(lines)


# ----------------------------------------------------------------------------
def html_to_text(raw: str) -> str:
    raw = re.sub(r"(?is)<(script|style|ix:header)[^>]*>.*?</\1>", " ", raw)
    raw = re.sub(r"(?s)<[^>]+>", " ", raw)
    return re.sub(r"\s+", " ", html.unescape(raw))


def ai_score(text: str) -> tuple[float, int, int]:
    """(mentions per 10,000 words, total mentions, word count)."""
    words = len(text.split())
    hits = sum(len(p.findall(text)) for p in AI_PATTERNS)
    return (hits / words * 10_000 if words else 0.0), hits, words


def tech_sic(sic) -> str | None:
    try:
        code = int(sic)
    except (TypeError, ValueError):
        return None
    for lo, hi, label in TECH_SIC_RANGES:
        if lo <= code <= hi:
            return label
    return None


def _growth_stats(fin: Financials) -> tuple[float, float]:
    rev = fin.annual.loc["revenue"].dropna()
    if len(rev) < 2:
        return np.nan, np.nan
    k = min(3, len(rev) - 1)
    cagr = (rev.iloc[-1] / rev.iloc[-1 - k]) ** (1 / k) - 1
    latest = rev.iloc[-1] / rev.iloc[-2] - 1
    return float(cagr), float(latest)


def classify(fin: Financials, filing_text: str | None = None) -> Classification:
    """Classify a company. Pass the 10-K text to enable the AI score."""
    t = THRESHOLDS
    a = fin.annual
    rev_last = a.loc["revenue"].dropna()
    rnd_last = a.loc["rnd"].dropna() if "rnd" in a.index else rev_last.iloc[0:0]
    rnd_int = float(rnd_last.iloc[-1] / rev_last.iloc[-1]) if len(rnd_last) and len(rev_last) else 0.0
    cagr, latest = _growth_stats(fin)

    reasons: list[str] = []
    sic_label = tech_sic(fin.sic)
    industry = (fin.industry or "").lower()
    if sic_label:
        tech_reason = f"SIC {fin.sic}: {sic_label}"
    elif fin.sic is None and (fin.sector == "Technology"
                              or any(w in industry for w in TECH_INDUSTRY_WORDS)):
        tech_reason = f"Yahoo sector/industry: {fin.sector} / {fin.industry}"
    elif rnd_int >= t["rnd_intensity_tech"]:
        tech_reason = f"R&D is {rnd_int:.1%} of revenue (>= {t['rnd_intensity_tech']:.0%})"
    else:
        tech_reason = None
    is_tech = tech_reason is not None
    if tech_reason is None:
        where = (f"SIC {fin.sic} ({fin.sic_description})" if fin.sic
                 else f"Industry '{fin.industry or 'unknown'}'")
        tech_reason_text = f"{where} is not a tech industry; R&D {rnd_int:.1%} of revenue"
    reasons.append(tech_reason or tech_reason_text)
    if np.isfinite(cagr):
        reasons.append(f"Revenue growth: {cagr:.1%} 3-yr CAGR, {latest:.1%} last year")

    score = mentions = None
    if filing_text:
        score, mentions, words = ai_score(filing_text)
        form = fin.latest_10k.form if fin.latest_10k else "annual report"
        reasons.append(f"AI score: {mentions} AI mentions in {words:,} words of the {form} "
                       f"= {score:.1f} per 10,000 words")
    ai_driven = score is not None and score >= t["ai_score_driven"]
    ai_exposed = score is not None and score >= t["ai_score_exposed"]

    if fin.is_financial:
        profile, model = "Financial", "standard"
        reasons.append("Banks/insurers need a dividend or excess-return model; standard DCF shown with caution")
    elif is_tech and (ai_driven or (np.isfinite(cagr) and cagr >= t["hyper_growth"])):
        profile, model = "AI / hyper-growth tech", "growth"
    elif is_tech and np.isfinite(cagr) and cagr >= t["growth"]:
        profile, model = "Growth tech", "growth"
    elif is_tech:
        profile, model = "Mature tech", "standard"
    else:
        profile, model = "Traditional", "standard"
        if ai_exposed:
            reasons.append("Mentions AI often, but is not a tech business -> standard model")

    return Classification(profile=profile, model=model, is_tech=is_tech, tech_reason=tech_reason,
                          rnd_intensity=rnd_int, revenue_cagr=cagr, latest_growth=latest,
                          ai_score=score, ai_mentions=mentions, reasons=reasons)


def fetch_10k_text(client, fin: Financials) -> str | None:
    """Download and clean the latest annual report (10-K or 20-F); None if unavailable."""
    if client is None or fin.latest_10k is None:
        return None
    try:
        return html_to_text(client.get_text(fin.latest_10k.url))
    except Exception:  # noqa: BLE001 - the AI score is optional
        return None


In [ ]:
%%writefile dcf/yahoo.py
"""
Fallback data source for companies that do not file with the SEC.

European companies listed only in Europe (e.g. LVMH ``MC.PA``, Siemens ``SIE.DE``,
Adyen ``ADYEN.AS``) never appear on EDGAR. For those, the standardized annual
statements published by Yahoo Finance are used instead (usually 4 fiscal years),
mapped onto exactly the same line items as the SEC parser so every model works unchanged.

``load_company`` is the single entry point: SEC first (US GAAP or IFRS), Yahoo otherwise.
"""
from __future__ import annotations

import re

import numpy as np
import pandas as pd

from .financials import (FLOW_TAGS, INSTANT_TAGS, PER_SHARE_TAGS, SHARE_FLOW_TAGS,
                         Financials, _derive, load_financials)
from .sec import SECClient, SECError

# Yahoo row labels -> standardized line items (first match wins).
YAHOO_MAP: dict[str, list[str]] = {
    "revenue": ["Total Revenue", "Operating Revenue"],
    "cogs": ["Cost Of Revenue"],
    "gross_profit": ["Gross Profit"],
    "rnd": ["Research And Development"],
    "sga": ["Selling General And Administration"],
    "operating_income": ["Operating Income", "Total Operating Income As Reported"],
    "interest_expense": ["Interest Expense", "Interest Expense Non Operating"],
    "pretax_income": ["Pretax Income"],
    "income_tax": ["Tax Provision"],
    "net_income": ["Net Income Common Stockholders", "Net Income"],
    "da": ["Reconciled Depreciation", "Depreciation And Amortization",
           "Depreciation Amortization Depletion"],
    "sbc": ["Stock Based Compensation"],
    "cfo": ["Operating Cash Flow", "Cash Flow From Continuing Operating Activities"],
    "capex": ["Capital Expenditure"],
    "dividends": ["Cash Dividends Paid", "Common Stock Dividend Paid"],
    "buybacks": ["Repurchase Of Capital Stock", "Common Stock Payments"],
    "eps_diluted": ["Diluted EPS"],
    "diluted_shares": ["Diluted Average Shares"],
    "cash": ["Cash And Cash Equivalents"],
    "st_investments": ["Other Short Term Investments"],
    "receivables": ["Accounts Receivable", "Receivables"],
    "inventory": ["Inventory"],
    "current_assets": ["Current Assets"],
    "lt_investments": ["Investments And Advances"],
    "ppe_net": ["Net PPE"],
    "total_assets": ["Total Assets"],
    "accounts_payable": ["Accounts Payable", "Payables"],
    "current_liabilities": ["Current Liabilities"],
    "debt_current_total": ["Current Debt", "Current Debt And Capital Lease Obligation"],
    "ltd_noncurrent": ["Long Term Debt", "Long Term Debt And Capital Lease Obligation"],
    "total_liabilities": ["Total Liabilities Net Minority Interest"],
    "equity": ["Stockholders Equity", "Common Stock Equity"],
    "minority_interest": ["Minority Interest"],
}

# Exchange suffixes that never exist on EDGAR -> go straight to Yahoo.
EXCHANGE_SUFFIX = re.compile(
    r"\.(AS|PA|DE|F|MI|MC|L|IL|SW|ST|CO|HE|OL|BR|LS|VI|IR|WA|AT|PR|BD|TO|V|HK|T|AX|SI|KS|NS|BO)$", re.I)

ALL_ITEMS = (list(FLOW_TAGS) + list(PER_SHARE_TAGS) + list(SHARE_FLOW_TAGS) + list(INSTANT_TAGS))


def has_exchange_suffix(ticker: str) -> bool:
    return bool(EXCHANGE_SUFFIX.search(ticker.strip()))


def financials_from_frames(ticker: str, income: pd.DataFrame, balance: pd.DataFrame,
                           cashflow: pd.DataFrame, info: dict, years: int = 5) -> Financials:
    """Build a Financials object from Yahoo statement tables (rows = labels, cols = dates)."""
    frames = [f for f in (income, balance, cashflow) if f is not None and not f.empty]
    if not frames:
        raise ValueError(f"Yahoo Finance has no financial statements for {ticker}.")
    combined = pd.concat(frames)
    combined = combined[~combined.index.duplicated(keep="first")]
    combined.columns = pd.to_datetime(combined.columns)
    combined = combined.reindex(sorted(combined.columns), axis=1)

    rows = {}
    for key in ALL_ITEMS:
        label = next((lbl for lbl in YAHOO_MAP.get(key, []) if lbl in combined.index), None)
        rows[key] = combined.loc[label].astype(float) if label else pd.Series(np.nan, index=combined.columns)
    annual = pd.DataFrame(rows).T
    annual = annual.loc[:, annual.loc["revenue"].notna()]          # drop empty years
    if annual.empty:
        raise ValueError(f"Yahoo Finance has no revenue history for {ticker}.")
    annual = annual.iloc[:, -years:]
    ends = annual.columns
    annual = _derive(annual)
    annual.columns = [f"FY{d.year}" for d in ends]

    shares = info.get("sharesOutstanding") or info.get("impliedSharesOutstanding")
    shares_date = "Yahoo Finance (latest)"
    if not shares:
        ordinary = combined.loc["Ordinary Shares Number"].dropna() if "Ordinary Shares Number" in combined.index else []
        shares = float(ordinary.iloc[-1]) if len(ordinary) else annual.loc["diluted_shares"].dropna().iloc[-1]
        shares_date = annual.columns[-1]

    ccy = info.get("financialCurrency") or info.get("currency") or "USD"
    warnings = [f"Not an SEC filer: statements from Yahoo Finance (standardized annual report data, {ccy})."]
    return Financials(
        ticker=ticker.upper(), name=info.get("longName") or info.get("shortName") or ticker.upper(),
        cik=0, sic=None, sic_description=info.get("industry"), annual=annual, ttm=None, ttm_end=None,
        shares_outstanding=float(shares), shares_date=shares_date, latest_10k=None,
        warnings=warnings, currency=ccy, source="Yahoo Finance", taxonomy="yahoo",
        sector=info.get("sector"), industry=info.get("industry"),
    )


def load_financials_yahoo(ticker: str, years: int = 5) -> Financials:
    import yfinance as yf
    t = yf.Ticker(ticker)
    try:
        info = t.info or {}
    except Exception:  # noqa: BLE001
        info = {}
    return financials_from_frames(ticker, t.income_stmt, t.balance_sheet, t.cashflow, info, years)


def load_company(ticker: str, user_agent: str | None = None, years: int = 5,
                 include_ttm: bool = True, source: str = "auto") -> tuple[Financials, SECClient | None]:
    """SEC EDGAR first (10-K or 20-F), Yahoo Finance as fallback. Returns (financials, sec_client)."""
    ticker = ticker.strip()
    note = None
    if source in ("auto", "sec") and not has_exchange_suffix(ticker):
        try:
            client = SECClient(user_agent)
            return load_financials(client, ticker, years, include_ttm), client
        except (SECError, ValueError) as exc:
            if source == "sec":
                raise
            note = f"SEC data not available ({exc})"
    fin = load_financials_yahoo(ticker, years)
    if note:
        fin.warnings.insert(0, note)
    return fin, None


In [ ]:
%%writefile dcf/growth.py
"""
Growth-stage DCF for AI and high-growth technology companies.

Why a standard 5-year DCF undervalues these companies
------------------------------------------------------
* Five years is too short: growth has to collapse to GDP-like rates by year 5.
* Regression betas of fast-growing stocks are high (often ~2), and a standard DCF
  keeps that high discount rate forever, even after the company has matured.
* CapEx % of revenue says nothing about how much a software / chip company must
  reinvest to grow (R&D, acquisitions, working capital, data centres...).

What this model does differently (Damodaran-style "young growth company" DCF)
-----------------------------------------------------------------------------
1. 10-year forecast in two stages. Growth starts at the analysts' consensus (or recent
   history), loses half of its excess over terminal growth during stage 1, then glides
   to terminal growth by the end of year 10.
2. Margins move from today's level to a target (e.g. a loss-making company scales up
   to a mature-tech margin).
3. Reinvestment is driven by a SALES-TO-CAPITAL ratio: every extra $1 of revenue needs
   $1 / (sales-to-capital) of new capital. Growth is never free.
4. The discount rate starts at today's WACC and converges to a MATURE WACC (beta -> 1)
   in stage 2, because a company growing 5% in year 10 is no longer a high-beta stock.
5. Terminal value uses a terminal ROIC: reinvestment rate = g / ROIC, so the perpetuity
   is consistent with how much the company reinvests.
6. Cross-checks that matter for tech: probability-weighted scenarios, a reverse DCF
   (growth the market is pricing in) and trading multiples.
"""
from __future__ import annotations

from dataclasses import dataclass, field, replace

import numpy as np
import pandas as pd

from .classify import Classification
from .financials import Financials
from .market import MarketData
from .model import (DCFResult, _clip, base_period, default_assumptions, run_dcf)

PRESETS = {
    "AI / hyper-growth tech": {"projection_years": 10, "stage1_years": 5,
                               "growth_cap": 0.50, "terminal_roic": 0.25},
    "Growth tech": {"projection_years": 10, "stage1_years": 3,
                    "growth_cap": 0.40, "terminal_roic": 0.18},
}
MATURE_TECH_MARGIN = 0.20     # target margin for companies that are not yet profitable
MARGIN_CEILING = 0.35         # margins above this attract competition: half the excess fades


@dataclass
class GrowthAssumptions:
    projection_years: int = 10
    stage1_years: int = 5
    initial_growth: float = 0.20
    terminal_growth: float = 0.025
    ebit_margin: float = 0.20               # base-period EBIT margin
    target_ebit_margin: float | None = None  # reached in the final year (None = hold)
    tax_rate: float = 0.21
    sales_to_capital: float = 1.5           # $ of new revenue per $1 of reinvestment
    terminal_roic: float = 0.20
    risk_free: float = 0.0425
    beta: float = 1.0
    mature_beta: float = 1.0
    erp: float = 0.045
    pre_tax_cost_of_debt: float = 0.055
    mature_wacc_override: float | None = None
    da_pct: float = 0.03                    # only for the implied EV/EBITDA check
    mid_year: bool = True
    include_lt_investments: bool = False
    sources: dict = field(default_factory=dict, repr=False)

    def growth_path(self) -> np.ndarray:
        n, s = self.projection_years, max(2, min(self.stage1_years, self.projection_years - 1))
        g0, gT = self.initial_growth, self.terminal_growth
        g_mid = gT + 0.5 * (g0 - gT)        # stage 1 removes half of the excess growth
        return np.interp(np.arange(1, n + 1), [1, s, n + 1], [g0, g_mid, gT])

    def margin_path(self) -> np.ndarray:
        target = self.ebit_margin if self.target_ebit_margin is None else self.target_ebit_margin
        return np.linspace(self.ebit_margin, target, self.projection_years + 1)[1:]


def _waccs(fin: Financials, mkt: MarketData, g: GrowthAssumptions, base: pd.Series) -> dict:
    equity = mkt.price * fin.shares_outstanding
    debt = float(np.nan_to_num(base.get("total_debt", 0.0)))
    we = equity / (equity + debt)
    kd_after = g.pre_tax_cost_of_debt * (1 - g.tax_rate)
    ke = g.risk_free + g.beta * g.erp
    ke_mature = g.risk_free + g.mature_beta * g.erp
    w0 = we * ke + (1 - we) * kd_after
    wm = g.mature_wacc_override if g.mature_wacc_override is not None else we * ke_mature + (1 - we) * kd_after
    return {"market_cap": equity, "debt": debt, "cost_of_equity": ke, "mature_cost_of_equity": ke_mature,
            "pre_tax_cost_of_debt": g.pre_tax_cost_of_debt, "after_tax_cost_of_debt": kd_after,
            "weight_equity": we, "weight_debt": 1 - we, "wacc": w0, "terminal_wacc": wm}


# ----------------------------------------------------------------------------
def growth_assumptions(fin: Financials, mkt: MarketData, cls: Classification | None = None,
                       use_ttm: bool = True, consensus_growth: float | None = None,
                       consensus_note: str = "", **overrides) -> GrowthAssumptions:
    """Defaults for the growth model, built on top of the standard assumptions."""
    std = default_assumptions(fin, mkt, use_ttm=use_ttm)
    base, _ = base_period(fin, use_ttm)
    preset = PRESETS.get(cls.profile if cls else "", PRESETS["Growth tech"])
    cap = preset["growth_cap"]
    src = {k: v for k, v in std.sources.items()}

    # Starting growth: analysts first, then recent history.
    rev = fin.annual.loc["revenue"].dropna()
    if consensus_growth is not None and np.isfinite(consensus_growth):
        g0 = _clip(consensus_growth, -0.05, cap, 0.15)
        src["initial_growth"] = f"{consensus_note} (cap {cap:.0%})"
    else:
        # Last year's growth, not the 3-yr CAGR: a boom 2-3 years ago should not be extrapolated.
        latest = rev.iloc[-1] / rev.iloc[-2] - 1 if len(rev) >= 2 else np.nan
        g0 = _clip(latest, -0.05, cap, 0.15)
        src["initial_growth"] = f"last fiscal year revenue growth {latest:.1%} (cap {cap:.0%})"

    # Margin: today's margin matters more than a 3-year average for a fast grower.
    m0 = base.get("ebit", np.nan) / base.get("revenue", np.nan)
    if np.isfinite(m0):
        src["ebit_margin"] = "base-period EBIT margin"
    else:
        m0 = std.ebit_margin
    target = None
    if m0 < 0.10:
        target = MATURE_TECH_MARGIN
        src["target_ebit_margin"] = f"low/negative margin scales to a {MATURE_TECH_MARGIN:.0%} mature-tech margin"
    elif m0 > MARGIN_CEILING:
        target = MARGIN_CEILING + 0.5 * (m0 - MARGIN_CEILING)
        src["target_ebit_margin"] = (f"competition erodes half of the margin above {MARGIN_CEILING:.0%} "
                                     "by the final year")
    else:
        src["target_ebit_margin"] = "held at today's margin"

    # Sales-to-capital = revenue / invested capital (equity + debt - cash).
    ic = (np.nan_to_num(base.get("equity", np.nan), nan=fin.annual.loc["equity"].iloc[-1])
          + np.nan_to_num(base.get("total_debt", 0.0))
          - np.nan_to_num(base.get("cash", 0.0)) - np.nan_to_num(base.get("st_investments", 0.0)))
    stc = base.get("revenue", np.nan) / ic if ic and ic > 0 else np.nan
    src["sales_to_capital"] = ("revenue / invested capital (equity + debt - cash), clipped 0.8..5"
                               if np.isfinite(stc) else "default 3.0 (invested capital not positive)")
    stc = _clip(stc, 0.8, 5.0, 3.0)

    src["terminal_roic"] = f"preset for '{cls.profile if cls else 'Growth tech'}' (moat persists)"
    src["mature_beta"] = "beta converges to 1.0 in stage 2 as the company matures"
    src["stage1_years"] = "preset: years of high growth before the fade"

    ga = GrowthAssumptions(
        projection_years=preset["projection_years"], stage1_years=preset["stage1_years"],
        initial_growth=g0, terminal_growth=std.terminal_growth, ebit_margin=float(m0),
        target_ebit_margin=target, tax_rate=std.tax_rate, sales_to_capital=stc,
        terminal_roic=preset["terminal_roic"], risk_free=std.risk_free, beta=std.beta,
        erp=std.erp, pre_tax_cost_of_debt=std.pre_tax_cost_of_debt, da_pct=std.da_pct,
        sources=src,
    )
    for k, v in overrides.items():
        if hasattr(ga, k) and getattr(ga, k) != v:
            src[k] = "manual override"
    return replace(ga, **overrides)


def run_growth_dcf(fin: Financials, mkt: MarketData, g: GrowthAssumptions,
                   use_ttm: bool = True) -> DCFResult:
    base, base_label = base_period(fin, use_ttm)
    warnings = list(fin.warnings)
    w = _waccs(fin, mkt, g, base)
    n, s, gT = g.projection_years, g.stage1_years, g.terminal_growth
    if w["terminal_wacc"] <= gT:
        raise ValueError(f"Mature WACC ({w['terminal_wacc']:.2%}) must exceed terminal growth ({gT:.2%}).")

    t = np.arange(1, n + 1)
    growth, margin = g.growth_path(), g.margin_path()
    revenue = base["revenue"] * np.cumprod(1 + growth)
    prev_rev = np.concatenate([[base["revenue"]], revenue[:-1]])
    ebit = revenue * margin
    taxes = ebit.clip(min=0) * g.tax_rate
    nopat = ebit - taxes
    reinvestment = (revenue - prev_rev) / g.sales_to_capital
    fcff = nopat - reinvestment

    # Year-by-year discount rate: today's WACC in stage 1, converging to mature WACC.
    rates = np.interp(t, [s, n], [w["wacc"], w["terminal_wacc"]])
    cum = np.cumprod(1 + rates)
    prev_cum = np.concatenate([[1.0], cum[:-1]])
    df = 1 / (prev_cum * (1 + rates) ** 0.5) if g.mid_year else 1 / cum
    pv = fcff * df

    # Terminal value with reinvestment consistent with growth: reinvestment rate = g / ROIC.
    nopat_next = revenue[-1] * (1 + gT) * margin[-1] * (1 - g.tax_rate)
    fcff_next = nopat_next * (1 - gT / g.terminal_roic)
    tv = fcff_next / (w["terminal_wacc"] - gT)
    pv_tv = tv / cum[-1]
    ev = pv.sum() + pv_tv

    cash = float(np.nan_to_num(base.get("cash", 0))) + float(np.nan_to_num(base.get("st_investments", 0)))
    lt_inv = float(np.nan_to_num(base.get("lt_investments", 0))) if g.include_lt_investments else 0.0
    minority = float(np.nan_to_num(base.get("minority_interest", 0)))
    equity = ev - w["debt"] + cash + lt_inv - minority
    per_share = equity / fin.shares_outstanding

    projection = pd.DataFrame({
        "Revenue": revenue, "Growth": growth, "EBIT": ebit, "EBIT margin": margin,
        "Taxes": -taxes, "NOPAT": nopat, "Reinvestment": -reinvestment, "FCFF": fcff,
        "WACC": rates, "Discount factor": df, "PV of FCFF": pv,
    }, index=[f"Y{i}" for i in t]).T

    if fcff[:3].sum() < 0:
        warnings.append("Free cash flow is negative in the early years - value depends on the "
                        "company reaching its target margin.")
    if pv_tv / ev > 0.85:
        warnings.append(f"Terminal value is {pv_tv / ev:.0%} of EV.")
    if g.terminal_roic < w["terminal_wacc"]:
        warnings.append("Terminal ROIC is below the mature WACC: growth destroys value in the terminal period.")

    ebitda_n = ebit[-1] + revenue[-1] * g.da_pct
    return DCFResult(
        company=fin.name, ticker=fin.ticker, base_label=base_label, projection=projection,
        wacc=w, enterprise_value=ev, pv_fcff=pv.sum(), pv_terminal=pv_tv, terminal_value=tv,
        bridge={"Enterprise value": ev, "Less: debt": -w["debt"],
                "Plus: cash & ST investments": cash, "Plus: LT investments": lt_inv,
                "Less: minority interest": -minority, "Equity value": equity},
        equity_value=equity, shares=fin.shares_outstanding, value_per_share=per_share,
        price=mkt.price, upside=per_share / mkt.price - 1,
        implied_exit_multiple=tv / ebitda_n if ebitda_n > 0 else np.nan,
        assumptions=g, warnings=warnings, model="growth", currency=fin.currency,
    )


# ----------------------------------------------------------------------------
# Cross-checks
# ----------------------------------------------------------------------------
def weighted_scenarios(fin, mkt, g: GrowthAssumptions, use_ttm=True,
                       probabilities=(0.25, 0.50, 0.25)) -> pd.DataFrame:
    """Bear / base / bull with probabilities, and the probability-weighted value."""
    final_margin = g.margin_path()[-1]
    cases = {
        "Bear": replace(g, initial_growth=g.initial_growth * 0.6 if g.initial_growth > 0 else g.initial_growth - 0.03,
                        target_ebit_margin=max(final_margin - 0.10, 0.05)),
        "Base": g,
        "Bull": replace(g, initial_growth=g.initial_growth * 1.25 if g.initial_growth > 0 else g.initial_growth + 0.03,
                        target_ebit_margin=final_margin + 0.05),
    }
    rows = {}
    for (name, case), p in zip(cases.items(), probabilities):
        res = run_growth_dcf(fin, mkt, case, use_ttm)
        rows[name] = {"Probability": p, "Y1 growth": case.initial_growth,
                      "Final EBIT margin": case.margin_path()[-1],
                      "Value / share": res.value_per_share, "Upside": res.upside}
    df = pd.DataFrame(rows).T
    weighted = float((df["Probability"] * df["Value / share"]).sum())
    df.loc["Probability-weighted"] = {"Probability": 1.0, "Y1 growth": np.nan,
                                      "Final EBIT margin": np.nan, "Value / share": weighted,
                                      "Upside": weighted / mkt.price - 1}
    return df


def growth_margin_grid(fin, mkt, g: GrowthAssumptions, use_ttm=True,
                       growth_step=0.05, margin_step=0.05, n=2) -> pd.DataFrame:
    """Value/share across starting growth (rows) x final EBIT margin (columns)."""
    final_margin = g.margin_path()[-1]
    gs = [round(g.initial_growth + i * growth_step, 4) for i in range(-n, n + 1)]
    ms = [round(final_margin + i * margin_step, 4) for i in range(-n, n + 1)]
    out = pd.DataFrame(index=gs, columns=ms, dtype=float)
    for gv in gs:
        for mv in ms:
            try:
                out.loc[gv, mv] = run_growth_dcf(
                    fin, mkt, replace(g, initial_growth=gv, target_ebit_margin=mv), use_ttm).value_per_share
            except ValueError:
                out.loc[gv, mv] = np.nan
    out.index.name, out.columns.name = "Starting revenue growth", "Final EBIT margin"
    return out


def implied_growth(fin, mkt, assumptions, use_ttm=True, lo=-0.10, hi=1.50, tol=1e-4) -> float | None:
    """Reverse DCF: the starting revenue growth at which value per share = market price."""
    runner = run_growth_dcf if isinstance(assumptions, GrowthAssumptions) else run_dcf

    def gap(g0):
        try:
            return runner(fin, mkt, replace(assumptions, initial_growth=g0), use_ttm).value_per_share - mkt.price
        except ValueError:
            return np.nan

    f_lo, f_hi = gap(lo), gap(hi)
    if not (np.isfinite(f_lo) and np.isfinite(f_hi)) or np.sign(f_lo) == np.sign(f_hi):
        return None
    for _ in range(80):
        mid = (lo + hi) / 2
        f_mid = gap(mid)
        if not np.isfinite(f_mid):
            return None
        if np.sign(f_mid) == np.sign(f_lo):
            lo, f_lo = mid, f_mid
        else:
            hi = mid
        if hi - lo < tol:
            break
    return (lo + hi) / 2


def trading_multiples(fin: Financials, mkt: MarketData, results: dict[str, DCFResult],
                      use_ttm=True) -> pd.DataFrame:
    """Market multiples today vs the multiples implied by each DCF."""
    base, _ = base_period(fin, use_ttm)
    cash = np.nan_to_num(base.get("cash", 0)) + np.nan_to_num(base.get("st_investments", 0))
    debt = np.nan_to_num(base.get("total_debt", 0))
    mcap = mkt.price * fin.shares_outstanding
    rev, ebit, ni = base.get("revenue", np.nan), base.get("ebit", np.nan), base.get("net_income", np.nan)
    ebitda = base.get("ebitda", np.nan)

    def row(ev, eq):
        return {"EV / Revenue": ev / rev, "EV / EBIT": ev / ebit,
                "EV / EBITDA": ev / ebitda if np.isfinite(ebitda) and ebitda > 0 else np.nan,
                "P / E": eq / ni if np.isfinite(ni) and ni > 0 else np.nan}

    cols = {"Market today": row(mcap + debt - cash, mcap)}
    for name, r in results.items():
        cols[name] = row(r.enterprise_value, r.equity_value)
    return pd.DataFrame(cols)


def rule_of_40(fin: Financials, cls: Classification) -> float | None:
    """Revenue growth + FCF margin (last fiscal year). >= 40% is the SaaS/tech benchmark."""
    a = fin.annual
    fcf_margin = a.loc["fcf"].iloc[-1] / a.loc["revenue"].iloc[-1]
    if not (np.isfinite(fcf_margin) and np.isfinite(cls.latest_growth)):
        return None
    return float(cls.latest_growth + fcf_margin)


In [ ]:
%%writefile dcf/report.py
"""Charts, formatted tables and Excel export for a DCF run."""
from __future__ import annotations


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

from .financials import Financials
from .model import CURRENCY_SYMBOLS, DCFResult

# Palette: blue = historical/positive, orange = projected/negative, warm-gray neutral.
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#b5b3ad"
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
DIVERGING = LinearSegmentedColormap.from_list("dcf_div", [ORANGE, "#f0efec", BLUE])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID,
    "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlecolor": INK, "legend.frameon": False,
})


def _millions(x, _pos=None):
    return f"{x / 1e6:,.0f}"


def _money(x: float, ccy: str = "USD", decimals: int = 2) -> str:
    """Currency amount safe for matplotlib text ('$' would otherwise start mathtext)."""
    sym = CURRENCY_SYMBOLS.get(ccy, ccy + " ").replace("$", "\\$")
    return f"{sym}{x:,.{decimals}f}"


# ----------------------------------------------------------------------------
# Charts
# ----------------------------------------------------------------------------
def plot_history_and_projection(fin: Financials, res: DCFResult):
    """Revenue and free cash flow: reported history vs. model projection (two panels)."""
    hist_rev = fin.annual.loc["revenue"]
    hist_fcf = fin.annual.loc["fcf"]
    proj = res.projection
    proj_labels = list(proj.columns)   # Y1..Yn from the base period (TTM or last FY)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, hist, fut, title in [
        (axes[0], hist_rev, proj.loc["Revenue"], "Revenue"),
        (axes[1], hist_fcf, proj.loc["FCFF"], "Free cash flow (hist: CFO - CapEx, proj: FCFF)"),
    ]:
        labels = list(hist.index) + proj_labels
        x = np.arange(len(labels))
        ax.bar(x[: len(hist)], hist.values, color=BLUE, width=0.6, label="Reported (10-K)")
        ax.bar(x[len(hist):], fut.values, color=ORANGE, width=0.6, label="Projected")
        ax.set_xticks(x, labels, rotation=45, ha="right")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(_millions))
        ax.set_ylabel(f"{fin.currency} millions")
        ax.set_title(title, loc="left")
        ax.axhline(0, color=INK_2, linewidth=0.8)
    axes[0].legend(loc="upper left")
    fig.suptitle(f"{res.company} ({res.ticker})", x=0.01, ha="left", color=INK, fontweight="bold")
    fig.tight_layout()
    return fig


def plot_bridge(res: DCFResult):
    """Waterfall from enterprise value to value per share."""
    items = [(k, v) for k, v in res.bridge.items() if k == "Equity value" or k == "Enterprise value" or abs(v) > 0]
    fig, ax = plt.subplots(figsize=(9, 4))
    running = 0.0
    for i, (label, val) in enumerate(items):
        total = label in ("Enterprise value", "Equity value")
        bottom = 0 if total else (running if val >= 0 else running + val)
        height = val if total else abs(val)
        color = BLUE if total else (GRAY if val >= 0 else ORANGE)
        ax.bar(i, height, bottom=bottom, color=color, width=0.6)
        ax.text(i, bottom + height, f"{val / 1e6:,.0f}", ha="center", va="bottom", fontsize=9, color=INK_2)
        running = val if total else running + val
    ax.set_xticks(range(len(items)), [k for k, _ in items], rotation=20, ha="right")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(_millions))
    ax.set_ylabel(f"{res.currency} millions")
    ax.set_title(f"EV to equity value: {_money(res.value_per_share, res.currency)} per share vs "
                 f"{_money(res.price, res.currency)} price ({res.upside:+.1%})", loc="left")
    fig.tight_layout()
    return fig


def plot_sensitivity(grid: pd.DataFrame, price: float, row_label="WACC", col_label="Terminal growth",
                     currency: str = "USD"):
    """Heatmap of value/share; blue above today's price, orange below, gray at the price."""
    vals = grid.values.astype(float)
    spread = np.nanmax(np.abs(vals - price)) or 1.0
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.imshow(vals, cmap=DIVERGING, vmin=price - spread, vmax=price + spread, aspect="auto")
    ax.grid(False)
    ax.set_xticks(range(grid.shape[1]), [f"{c:.1%}" for c in grid.columns])
    ax.set_yticks(range(grid.shape[0]), [f"{r:.1%}" for r in grid.index])
    ax.set_xlabel(col_label)
    ax.set_ylabel(row_label)
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            if np.isfinite(vals[i, j]):
                ax.text(j, i, _money(vals[i, j], currency, 0), ha="center", va="center", fontsize=9, color=INK)
    ci, cj = vals.shape[0] // 2, vals.shape[1] // 2      # base case sits in the centre
    ax.add_patch(plt.Rectangle((cj - 0.5, ci - 0.5), 1, 1, fill=False, edgecolor=INK, linewidth=1.5))
    ax.set_title(f"Value per share sensitivity (current price {_money(price, currency)}, base case boxed)",
                 loc="left")
    fig.tight_layout()
    return fig


# ----------------------------------------------------------------------------
# Tables
# ----------------------------------------------------------------------------
def assumptions_table(res: DCFResult) -> pd.DataFrame:
    if res.model == "growth":
        return growth_assumptions_table(res)
    a = res.assumptions
    pct = lambda x: f"{x:.2%}"  # noqa: E731
    rows = [
        ("Year-1 revenue growth", pct(a.initial_growth), a.sources.get("initial_growth", "")),
        ("Terminal growth", pct(a.terminal_growth), a.sources.get("terminal_growth", "")),
        ("EBIT margin (base)", pct(a.ebit_margin), a.sources.get("ebit_margin", "")),
        ("EBIT margin (final year)", pct(a.margin_path()[-1]),
         "target override" if a.target_ebit_margin is not None else "held flat"),
        ("Tax rate", pct(a.tax_rate), a.sources.get("tax_rate", "")),
        ("D&A % revenue", pct(a.da_pct), a.sources.get("da_pct", "")),
        ("CapEx % revenue", pct(a.capex_pct), a.sources.get("capex_pct", "")),
        ("NWC % revenue", pct(a.nwc_pct), a.sources.get("nwc_pct", "")),
        ("Risk-free rate", pct(a.risk_free), a.sources.get("risk_free", "")),
        ("Beta", f"{a.beta:.2f}", a.sources.get("beta", "")),
        ("Equity risk premium", pct(a.erp), a.sources.get("erp", "")),
        ("Cost of equity", pct(res.wacc["cost_of_equity"]), "CAPM: rf + beta x ERP"),
        ("Pre-tax cost of debt", pct(a.pre_tax_cost_of_debt), a.sources.get("pre_tax_cost_of_debt", "")),
        ("Equity / debt weights", f"{res.wacc['weight_equity']:.0%} / {res.wacc['weight_debt']:.0%}",
         "market cap vs book debt"),
        ("WACC", pct(res.wacc["wacc"]),
         "manual override" if a.wacc_override is not None else "computed"),
        ("Discounting", "mid-year" if a.mid_year else "end-of-year", ""),
    ]
    return pd.DataFrame(rows, columns=["Assumption", "Value", "Source"]).set_index("Assumption")


def growth_assumptions_table(res: DCFResult) -> pd.DataFrame:
    a, w = res.assumptions, res.wacc
    pct = lambda x: f"{x:.2%}"  # noqa: E731
    src = a.sources.get
    rows = [
        ("Forecast length", f"{a.projection_years} years ({a.stage1_years} high-growth)", src("stage1_years", "")),
        ("Year-1 revenue growth", pct(a.initial_growth), src("initial_growth", "")),
        (f"Growth at end of year {a.stage1_years}", pct(a.growth_path()[a.stage1_years - 1]),
         "half of the excess over terminal growth is gone"),
        ("Terminal growth", pct(a.terminal_growth), src("terminal_growth", "")),
        ("EBIT margin (base)", pct(a.ebit_margin), src("ebit_margin", "")),
        ("EBIT margin (final year)", pct(a.margin_path()[-1]), src("target_ebit_margin", "")),
        ("Tax rate", pct(a.tax_rate), src("tax_rate", "")),
        ("Sales-to-capital", f"{a.sales_to_capital:.2f}x", src("sales_to_capital", "")),
        ("Terminal ROIC", pct(a.terminal_roic), src("terminal_roic", "")),
        ("Terminal reinvestment rate", pct(a.terminal_growth / a.terminal_roic), "g / ROIC"),
        ("Risk-free rate", pct(a.risk_free), src("risk_free", "")),
        ("Beta today -> mature", f"{a.beta:.2f} -> {a.mature_beta:.2f}", src("mature_beta", "")),
        ("Equity risk premium", pct(a.erp), src("erp", "")),
        ("Cost of equity today -> mature", f"{pct(w['cost_of_equity'])} -> {pct(w['mature_cost_of_equity'])}",
         "CAPM: rf + beta x ERP"),
        ("Pre-tax cost of debt", pct(a.pre_tax_cost_of_debt), src("pre_tax_cost_of_debt", "")),
        ("WACC today -> mature", f"{pct(w['wacc'])} -> {pct(w['terminal_wacc'])}",
         "manual override (mature)" if a.mature_wacc_override is not None else "converges in stage 2"),
        ("Discounting", "mid-year" if a.mid_year else "end-of-year", ""),
    ]
    return pd.DataFrame(rows, columns=["Assumption", "Value", "Source"]).set_index("Assumption")


def projection_table(res: DCFResult) -> pd.DataFrame:
    """Projection with $ rows in millions and ratio rows as percentages (strings for display)."""
    p = res.projection.copy().astype(object)
    for idx in p.index:
        if idx in ("Growth", "EBIT margin", "WACC"):
            p.loc[idx] = [f"{v:.1%}" for v in res.projection.loc[idx]]
        elif idx == "Discount factor":
            p.loc[idx] = [f"{v:.3f}" for v in res.projection.loc[idx]]
        else:
            p.loc[idx] = [f"{v / 1e6:,.0f}" for v in res.projection.loc[idx]]
    return p


def ratios_table(fin: Financials) -> pd.DataFrame:
    a = fin.annual
    def safe(num, den):
        return (a.loc[num] / a.loc[den]).replace([np.inf, -np.inf], np.nan)
    df = pd.DataFrame({
        "Revenue growth": a.loc["revenue"].pct_change(fill_method=None),
        "Gross margin": safe("gross_profit", "revenue"),
        "EBIT margin": safe("ebit", "revenue"),
        "Net margin": safe("net_income", "revenue"),
        "Effective tax rate": a.loc["tax_rate"],
        "CapEx % revenue": safe("capex", "revenue"),
        "FCF margin": safe("fcf", "revenue"),
        "ROE": safe("net_income", "equity"),
        "Debt / EBITDA": safe("total_debt", "ebitda"),
    }).T
    return df


def style_statement(df: pd.DataFrame):
    """Notebook styling: $m with thousands separators, EPS with 2 decimals."""
    sty = df.style.format("{:,.0f}", na_rep="–")
    eps = [i for i in df.index if "EPS" in i]
    if eps:
        sty = sty.format("{:,.2f}", na_rep="–", subset=pd.IndexSlice[eps, :])
    return sty


def style_ratios(df: pd.DataFrame):
    sty = df.style.format("{:.1%}", na_rep="–")
    if "Debt / EBITDA" in df.index:
        sty = sty.format("{:.2f}x", na_rep="–", subset=pd.IndexSlice[["Debt / EBITDA"], :])
    return sty


# ----------------------------------------------------------------------------
# Excel
# ----------------------------------------------------------------------------
def export_excel(fin: Financials, res: DCFResult, grid: pd.DataFrame | None = None,
                 path: str | None = None, extra_sheets: dict | None = None) -> str:
    path = path or f"{res.ticker}_DCF.xlsx"
    with pd.ExcelWriter(path, engine="openpyxl") as xl:
        res.summary().to_frame("Value").to_excel(xl, sheet_name="Summary")
        assumptions_table(res).to_excel(xl, sheet_name="Assumptions")
        (res.projection / pd.Series(
            {i: (1 if i in ("Growth", "EBIT margin", "WACC", "Discount factor") else 1e6)
             for i in res.projection.index}).values[:, None]
         ).to_excel(xl, sheet_name=f"Projection ({res.currency}m)")
        for name in ("Income Statement", "Balance Sheet", "Cash Flow"):
            fin.statement(name).to_excel(xl, sheet_name=f"{name} ({fin.currency}m)"[:31])
        ratios_table(fin).to_excel(xl, sheet_name="Ratios")
        if grid is not None:
            g = grid.copy()
            rname = "WACC" if g.index.name in (None, "wacc_override") else g.index.name
            cname = "g" if g.columns.name in (None, "terminal_growth") else g.columns.name
            g.index = [f"{rname} {w:.2%}" for w in g.index]
            g.columns = [f"{cname} {c:.2%}" for c in g.columns]
            g.to_excel(xl, sheet_name="Sensitivity")
        for name, df in (extra_sheets or {}).items():
            df.to_excel(xl, sheet_name=name[:31])
        for ws in xl.book.worksheets:
            ws.column_dimensions["A"].width = 34
            for col in "BCDEFGHIJKL":
                ws.column_dimensions[col].width = 15
    return path


In [ ]:
import numpy as np, pandas as pd
from IPython.display import display, Markdown
import dcf
from dcf import (SECClient, load_financials, fetch_market_data, default_assumptions, run_dcf,
                 wacc_growth_grid, scenarios, classify, fetch_10k_text, growth_assumptions,
                 run_growth_dcf, weighted_scenarios, growth_margin_grid, implied_growth,
                 trading_multiples, rule_of_40)
from dcf.market import fetch_consensus_growth
from dcf.model import money, CURRENCY_SYMBOLS
from dcf import report
pd.set_option("display.max_colwidth", 140)
print(f"dcf v{dcf.__version__} ready")

In [ ]:
#@title 1 · Inputs
TICKER = "NVDA"  #@param {type:"string"}
#@markdown The SEC requires a name and e-mail in every request:
SEC_USER_AGENT = "Your Name your.email@example.com"  #@param {type:"string"}
USE_TTM = True  #@param {type:"boolean"}
#@markdown `auto` lets the classifier choose; or force a model:
MODEL = "auto"  #@param ["auto", "growth", "standard"]
#@markdown Download the latest 10-K and count AI mentions (adds a few seconds):
SCAN_10K_FOR_AI = True  #@param {type:"boolean"}
YEARS_OF_HISTORY = 5  #@param {type:"slider", min:3, max:10, step:1}
#@markdown **European / non-US companies:** use the US ticker if it is listed in the US (ASML, SAP, NVO); otherwise the home-exchange ticker (ASML.AS, MC.PA, SIE.DE, NESN.SW).
#@markdown Risk-free rate as a decimal. None = US 10Y Treasury. For EUR companies enter the German 10Y Bund yield:
RISK_FREE = None  #@param {type:"raw"}

## 2 · Financial statements from the SEC

In [ ]:
fin, client = dcf.load_company(TICKER, SEC_USER_AGENT, years=YEARS_OF_HISTORY, include_ttm=USE_TTM)
SYM = CURRENCY_SYMBOLS.get(fin.currency, fin.currency + " ")
lines = [f"### {fin.name} ({fin.ticker})",
         f"Source: **{fin.source}** · statements in **{fin.currency}**"]
if fin.sic:
    lines.append(f"CIK {fin.cik} · SIC {fin.sic} – {fin.sic_description}")
elif fin.industry:
    lines.append(f"Sector: {fin.sector} · Industry: {fin.industry}")
if fin.latest_10k:
    k = fin.latest_10k
    lines.append(f"Latest {k.form}: period ending **{k.report_date}**, filed {k.filing_date} – [open the filing]({k.url})")
if fin.ttm is not None:
    lines.append(f"TTM column from the 10-Q for the period ending **{fin.ttm_end}**")
lines.append(f"Shares outstanding: {fin.shares_outstanding/1e6:,.1f}m (as of {fin.shares_date})")
display(Markdown("  \n".join(lines)))
for w in fin.warnings:
    print("⚠️", w)
for name in ("Income Statement", "Balance Sheet", "Cash Flow"):
    display(Markdown(f"**{name}** ({fin.currency} millions)"))
    display(report.style_statement(fin.statement(name)))
display(Markdown("**Key ratios**"))
display(report.style_ratios(report.ratios_table(fin)))

## 3 · What kind of company is this?

In [ ]:
text = fetch_10k_text(client, fin) if SCAN_10K_FOR_AI else None
if SCAN_10K_FOR_AI and text is None:
    print("Could not download the 10-K text - classifying without the AI score.")
cls = classify(fin, text)
model_choice = cls.model if MODEL == "auto" else MODEL
r40 = rule_of_40(fin, cls)
display(Markdown(cls.summary()))
if r40 is not None:
    display(Markdown(f"- Rule of 40 (revenue growth + FCF margin): **{r40:.0%}** "
                     f"({'passes' if r40 >= 0.40 else 'below'} the 40% tech benchmark)"))
display(Markdown(f"**Model used: {model_choice} DCF**" + ("" if MODEL == "auto" else " (forced)")))

## 4 · Market data

In [ ]:
mkt = fetch_market_data(fin.ticker, reporting_currency=fin.currency, risk_free=RISK_FREE)
extra = f" (= {mkt.price_original:,.2f} {mkt.price_currency})" if mkt.price_currency != fin.currency else ""
print(f"Price: {money(mkt.price, fin.currency)}{extra}  ({mkt.price_date})")
print(f"Market cap: {money(mkt.price * fin.shares_outstanding / 1e9, fin.currency, 1)}bn")
consensus, consensus_note = fetch_consensus_growth(fin.ticker)
for n in mkt.notes:
    print("•", n)
print("•", consensus_note if consensus is None else f"Revenue growth: {consensus_note}")

## 5 · Assumptions
Every default is derived from the filings or the market. Type a decimal to override (`0.08` = 8%), or leave `None`.

In [ ]:
#@title Overrides (None = derived default)
#@markdown **Both models**
INITIAL_GROWTH = None        #@param {type:"raw"}
TERMINAL_GROWTH = 0.025      #@param {type:"raw"}
TARGET_EBIT_MARGIN = None    #@param {type:"raw"}
EQUITY_RISK_PREMIUM = 0.045  #@param {type:"raw"}
BETA = None                  #@param {type:"raw"}
#@markdown **Growth-stage model (AI / tech)**
HIGH_GROWTH_YEARS = None     #@param {type:"raw"}
TERMINAL_ROIC = None         #@param {type:"raw"}
SALES_TO_CAPITAL = None      #@param {type:"raw"}
MATURE_WACC = None           #@param {type:"raw"}
#@markdown **Standard model**
WACC_OVERRIDE = None         #@param {type:"raw"}

def given(d):
    return {k: v for k, v in d.items() if v is not None}

common = given({"initial_growth": INITIAL_GROWTH, "terminal_growth": TERMINAL_GROWTH,
                "target_ebit_margin": TARGET_EBIT_MARGIN, "erp": EQUITY_RISK_PREMIUM, "beta": BETA})
g_only = given({"stage1_years": HIGH_GROWTH_YEARS, "terminal_roic": TERMINAL_ROIC,
                "sales_to_capital": SALES_TO_CAPITAL, "mature_wacc_override": MATURE_WACC})
s_only = given({"wacc_override": WACC_OVERRIDE})

std_a = default_assumptions(fin, mkt, use_ttm=USE_TTM, **common, **s_only)
std_res = run_dcf(fin, mkt, std_a, use_ttm=USE_TTM)
gro_a = growth_assumptions(fin, mkt, cls, USE_TTM, consensus, consensus_note, **common, **g_only)
gro_res = run_growth_dcf(fin, mkt, gro_a, use_ttm=USE_TTM)

result = gro_res if model_choice == "growth" else std_res
assumptions = gro_a if model_choice == "growth" else std_a
display(report.assumptions_table(result))

## 6 · DCF result

In [ ]:
display(Markdown(f"**Projection** ({fin.currency} millions, base period: {result.base_label})"))
display(report.projection_table(result))
display(result.summary().to_frame("Value"))
for w in result.warnings:
    print("⚠️", w)

In [ ]:
report.plot_bridge(result);

In [ ]:
report.plot_history_and_projection(fin, result);

## 7 · Standard vs growth model vs the market
The **reverse DCF** solves for the starting revenue growth that makes each model's value equal today's price.
If the standard model needs an implausible growth rate but the growth model needs a realistic one, the
standard model's structure (5 years, constant high beta) is what was holding the value down.

In [ ]:
rows = {}
for name, res, a in [("Standard DCF", std_res, std_a), ("Growth-stage DCF", gro_res, gro_a)]:
    ig = implied_growth(fin, mkt, a, use_ttm=USE_TTM)
    rows[name] = {"Value / share": res.value_per_share, "Upside": res.upside,
                  "Starting growth used": a.initial_growth,
                  "Growth the market implies": ig if ig is not None else np.nan,
                  "Terminal value % EV": res.terminal_share_of_ev}
comparison = pd.DataFrame(rows).T
display(comparison.style.format({"Value / share": SYM + "{:,.2f}", "Upside": "{:+.1%}",
                                 "Starting growth used": "{:.1%}", "Growth the market implies": "{:.1%}",
                                 "Terminal value % EV": "{:.0%}"}, na_rep="n/a"))
multiples = trading_multiples(fin, mkt, {"Standard DCF": std_res, "Growth-stage DCF": gro_res}, USE_TTM)
display(Markdown("**Trading multiples: market today vs implied by each model**"))
display(multiples.style.format("{:.1f}x", na_rep="–"))

## 8 · Sensitivity and scenarios

In [ ]:
if model_choice == "growth":
    grid = growth_margin_grid(fin, mkt, gro_a, use_ttm=USE_TTM)
    report.plot_sensitivity(grid, mkt.price, "Starting revenue growth", "Final EBIT margin", fin.currency)
    sc = weighted_scenarios(fin, mkt, gro_a, use_ttm=USE_TTM)
    fmt = {"Probability": "{:.0%}", "Y1 growth": "{:.1%}", "Final EBIT margin": "{:.1%}",
           "Value / share": SYM + "{:,.2f}", "Upside": "{:+.1%}"}
else:
    grid = wacc_growth_grid(fin, mkt, std_a, std_res, use_ttm=USE_TTM)
    report.plot_sensitivity(grid, mkt.price, currency=fin.currency)
    sc = scenarios(fin, mkt, std_a, use_ttm=USE_TTM)
    fmt = {"Y1 growth": "{:.1%}", "Final EBIT margin": "{:.1%}", "Value / share": SYM + "{:,.2f}", "Upside": "{:+.1%}"}
display(sc.style.format(fmt, na_rep=""))

## 9 · Export to Excel

In [ ]:
cls_table = pd.DataFrame({"Value": [cls.profile, model_choice] + cls.reasons},
                         index=["Profile", "Model"] + [f"Reason {i+1}" for i in range(len(cls.reasons))])
path = report.export_excel(fin, result, grid, extra_sheets={
    "Classification": cls_table, "Model comparison": comparison,
    "Multiples": multiples, "Scenarios": sc})
print("Saved", path)
try:
    from google.colab import files
    files.download(path)
except ImportError:
    pass

## How the two models differ

| | Standard DCF | Growth-stage DCF (AI / tech) |
|---|---|---|
| Forecast | 5 years | 10 years in two stages |
| Starting growth | 3-yr revenue CAGR, capped at 25% | analyst consensus (else last year's growth), capped at 50% (AI) / 40% (growth tech) |
| Growth path | straight line to terminal growth | keeps half of its excess growth through stage 1, then glides to terminal growth |
| Margin | 3-yr average, held flat | today's margin; above 35% half the excess fades (competition), below 10% scales up to 20% |
| Reinvestment | D&A, CapEx and working capital as % of revenue | sales-to-capital: each extra $1 of revenue needs $1 / (sales-to-capital) of capital |
| Discount rate | one WACC forever | today's WACC in stage 1, falls to a mature WACC (beta 1.0) by year 10 |
| Terminal value | FCFF × (1+g) / (WACC − g) | NOPAT × (1 − g / ROIC) / (mature WACC − g) |
| Extra checks | WACC × g grid, bear/base/bull | growth × margin grid, probability-weighted scenarios |

**Classification thresholds** (in `dcf/classify.py`, `THRESHOLDS`): R&D ≥ 8% of revenue counts as tech; ≥ 20 AI
mentions per 10,000 words in the 10-K counts as AI-driven; 3-yr revenue CAGR ≥ 30% is hyper-growth, ≥ 10% is growth.
These are judgement calls - run a few known companies to check they behave as you expect.

*Educational model, not investment advice.*